# Circuit Topology predict free energy of folding, unfolding- and folding rate

In this code Circuit Topology will be used to predict the free energy of folding, folding- and unfolding rate of proteins from the KPRO database. Firstly, a dataframe is prepared with kinetic information. Here the free energy of folding is calculated. Secondly, a linear regression model is built. This model takes CT parameters as input as well as contact order, for comparison. After, the model is validated using ACPRO data for the folding rate. The predicted free energy of folding is compared to the calculated predictions. Lastly, the model is used to make predictions on the MERGED database (PED and SCOPe).

## Import libraries

In [2]:
import numpy as np
import os
import pandas as pd

from scipy import ndimage
from scipy.signal import savgol_filter
from scipy.stats import linregress
from sklearn.model_selection import train_test_split

import matplotlib.pyplot as plt

import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from plotly.colors import sample_colorscale

from sklearn.metrics import r2_score
import statsmodels.api as sm
from scipy import stats
import mdtraj as md
import MDAnalysis as mda
import itertools

from sklearn.decomposition import PCA
import ast
import plotly.io as pio

## Functions

In [3]:
#Function to calculate the contact order of a protein. 
#The contact order is a metric to define the distance between residues that form contacts with each other. 
#It is used to describe folding or compactness of a protein
# Small CO: 0.02-0.05, large CO: 0.1-0.2

def abs_contact_order3(pdb, cutoff_nm=.6):
    """Return the absolute contact order."""

    traj = md.load(pdb)
    print("Atoms: ", traj.n_atoms)
    seq_atoms = np.array([a.residue.resSeq for a in traj.top.atoms], dtype=int)
    xyz = traj.xyz[0]
    
        
    contact_count = 0
    seq_distance_sum = 0
    cutoff_2 = cutoff_nm*cutoff_nm
    N = len(seq_atoms)
    for i in range(N):
        for j in range(i+1,N):
            seq_dist = seq_atoms[j] - seq_atoms[i]
            if seq_dist > 0:
                d = 0.0
                for k in range(3):
                    d += (xyz[j,k] - xyz[i,k])**2
                if d < cutoff_2:
                    seq_distance_sum += 2*seq_dist 
                    contact_count += 2


    if contact_count==0.:
        return 0.
    print("L: ", traj.n_residues)

    return seq_distance_sum/float(contact_count), traj.n_residues


work_dir_path = r"C:\Users\akulovve\Documents\MS_student\Muriel\paper\code"

## Wotrk with K-Pro database 

### Make DataFrame of proteins, fragments, k_f, k_u, T, dG, Two_State

In [ ]:
#read kon and koff data
df_kinetics = pd.read_csv(f'{work_dir_path}\KPro.tsv', sep='\t', header=0)


df_kinetics = df_kinetics[["PDB_wild","MUTATION_PDB", "MUTATED_CHAIN","T","ln(kf)_H2O","ln(ku)_H2O"]]

print("Filter 1: ", df_kinetics["PDB_wild"].unique().shape)

df_kinetics = df_kinetics.loc[df_kinetics["MUTATION_PDB"]=="WT"]

print("Filter 2: ", df_kinetics["PDB_wild"].unique().shape)

df_kinetics["T"] += 273.15
df_kinetics["K_diff"] = (df_kinetics["ln(kf)_H2O"]-df_kinetics["ln(ku)_H2O"])
df_kinetics["Energy, kcal/mol"] = 1.987204258640 * df_kinetics["T"] * df_kinetics["K_diff"] / 1000 #in R = 1.987204258640 cal/K/mol 
df_kinetics['log(Energy, kcal/mol)'] = np.log(df_kinetics['Energy, kcal/mol'])

print("Filter 3: ", df_kinetics["PDB_wild"].unique().shape)
df_kinetics.dropna(inplace=True)

print("Filter 4: ", df_kinetics["PDB_wild"].unique().shape)

df_kinetics[df_kinetics["PDB_wild"] == "1prs"]

df_kinetics.to_csv(f'{work_dir_path}\KPro_VA_v1.csv')


Filter 1:  (65,)
Filter 2:  (57,)
Filter 3:  (57,)
Filter 4:  (54,)


c:\Users\akulovve\Documents\MS_student\Muriel\paper\code\.conda\Lib\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)


In [ ]:
# Manually add Two_state and Fragment Start, Fragment End 

,PDB_wild,MUTATION_PDB,MUTATED_CHAIN,T,ln(kf)_H2O,ln(ku)_H2O,STATE,K_diff,"Energy, kcal/mol","log(Energy, kcal/mol)"
0,1azu,WT,A,298.15,4.84,-16.96,2,21.80,12.916172,2.558480
1,1ba5,WT,A,298.15,5.91,1.16,Yes,4.75,2.814304,1.034715
2,1bdc,WT,A,310.15,11.70,4.22,Yes,7.48,4.610159,1.528262
6,1c9o,WT,A,298.15,7.22,-0.45,Yes,7.67,4.544360,1.513887
7,1coa,WT,I,298.15,4.03,-9.04,2,13.07,7.743778,2.046890
...,...,...,...,...,...,...,...,...,...,...
83,2vwf,WT,A,295.15,2.77,-1.97,Yes,4.74,2.780121,1.022494
84,2vxd,WT,A,283.15,7.00,1.06,Yes,5.94,3.342301,1.206659
88,3kz3,WT,A,298.15,10.38,3.21,Yes,7.17,4.248117,1.446476
89,AF-P14621,WT,A,301.15,-1.42,-9.64,2,8.22,4.919231,1.593152


### Load new dataframe

In [6]:
df_kinetics = pd.read_csv(f'{work_dir_path}\KPro_VA_v2.csv', index_col=0)
print(df_kinetics.columns)

df_kinetics

Index(['PDB_wild', 'MUTATION_PDB', 'MUTATED_CHAIN', 'T', 'ln(kf)_H2O',
       'ln(ku)_H2O', 'K_diff', 'Energy, kcal/mol', 'log(Energy, kcal/mol)',
       'Folding Type', 'Fragment Start', 'Fragment End'],
      dtype='object')


,PDB_wild,MUTATION_PDB,MUTATED_CHAIN,T,ln(kf)_H2O,ln(ku)_H2O,K_diff,"Energy, kcal/mol","log(Energy, kcal/mol)",Folding Type,Fragment Start,Fragment End
0,1azu,WT,A,298.15,4.84,-16.96,21.80,12.916172,2.558480,Two,NaN,NaN
1,1ba5,WT,A,298.15,5.91,1.16,4.75,2.814304,1.034715,Two,NaN,NaN
2,1bdc,WT,A,310.15,11.70,4.22,7.48,4.610159,1.528262,Two,NaN,NaN
6,1c9o,WT,A,298.15,7.22,-0.45,7.67,4.544360,1.513887,Two,NaN,NaN
7,1coa,WT,I,298.15,4.03,-9.04,13.07,7.743778,2.046890,Two,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
97,1AYI,WT,A,298.15,7.20,2.34,4.86,2.879477,1.057609,Two,NaN,NaN
99,1APS,WT,A,298.15,-1.58,-9.00,7.42,4.396238,1.480749,Two,NaN,NaN
102,1FHT,WT,A,298.15,4.62,-11.72,16.34,9.681204,2.270186,Two,2.0,102.0
103,1UBQ,WT,A,298.15,7.33,-6.84,14.17,8.395512,2.127697,Two,NaN,NaN


### Dowload pdbs

In [32]:
import urllib

for pdb in df_kinetics.PDB_wild.unique():
    if not os.path.isfile('pdbs/{}.pdb'.format(pdb)):
        try:
            urllib.request.urlretrieve('https://files.rcsb.org/download/{}.pdb'.format(pdb.lower()), 'pdbs/{}.pdb'.format(pdb.lower()))
        except:
            print(f"Problems with {pdb}")

Problems with AF-P14621


In [33]:
df_kinetics

,PDB_wild,MUTATION_PDB,MUTATED_CHAIN,T,ln(kf)_H2O,ln(ku)_H2O,K_diff,"Energy, kcal/mol","log(Energy, kcal/mol)",Folding Type,Fragment Start,Fragment End
0,1azu,WT,A,298.15,4.84,-16.96,21.80,12.916172,2.558480,Two,NaN,NaN
1,1ba5,WT,A,298.15,5.91,1.16,4.75,2.814304,1.034715,Two,NaN,NaN
2,1bdc,WT,A,310.15,11.70,4.22,7.48,4.610159,1.528262,Two,NaN,NaN
6,1c9o,WT,A,298.15,7.22,-0.45,7.67,4.544360,1.513887,Two,NaN,NaN
7,1coa,WT,I,298.15,4.03,-9.04,13.07,7.743778,2.046890,Two,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
97,1AYI,WT,A,298.15,7.20,2.34,4.86,2.879477,1.057609,Two,NaN,NaN
99,1APS,WT,A,298.15,-1.58,-9.00,7.42,4.396238,1.480749,Two,NaN,NaN
102,1FHT,WT,A,298.15,4.62,-11.72,16.34,9.681204,2.270186,Two,2.0,102.0
103,1UBQ,WT,A,298.15,7.33,-6.84,14.17,8.395512,2.127697,Two,NaN,NaN


In [ ]:
import os
import pandas as pd
import numpy as np

def is_hydrogen(atom_name):
    return atom_name.startswith('H')

def extract_chain_from_pdb_lines(lines, target_chain_id, frag_start=None, frag_end=None):
    """
    Extracts lines for a specific chain (excluding hydrogens) from a list of PDB lines.
    If frag_start and frag_end are provided, only extracts residues within that range.
    Returns a list of frames (each frame is a list of lines).
    """
    frames = []
    current_frame = []
    inside_model = False

    for line in lines:
        record = line[:6].strip()

        if record == 'MODEL':
            inside_model = True
            current_frame = []
        elif record == 'ENDMDL':
            inside_model = False
            frames.append(current_frame)
        elif record == 'ATOM':
            chain_id = line[21].strip()
            atom_name = line[12:16].strip()
            res_seq = int(line[22:26].strip())

            if chain_id == target_chain_id and not is_hydrogen(atom_name):
                if frag_start is not None and frag_end is not None:
                    if not (frag_start <= res_seq <= frag_end):
                        continue  # Skip residues outside fragment range

                if not inside_model:
                    current_frame = []  # Start frame on-the-fly if no MODEL/ENDMDL
                    inside_model = True

                current_frame.append(line)

        elif record in ('TER', 'END') and inside_model:
            current_frame.append(line)

    if inside_model and current_frame:
        frames.append(current_frame)

    return frames

def process_pdbs_from_dataframe_simple(input_folder, output_folder, dataframe):
    os.makedirs(output_folder, exist_ok=True)
    df = dataframe.copy()

    df['PDB_wild'] = df['PDB_wild'].str.lower().str.strip()
    df['MUTATED_CHAIN'] = df['MUTATED_CHAIN'].str.strip()

    saved_files = []  # Track intermediate files

    for _, row in df.iterrows():
        pdb_name = row['PDB_wild']
        chain_id = row['MUTATED_CHAIN']
        frag_start = row.get("Fragment Start", np.nan)
        frag_end = row.get("Fragment End", np.nan)

        # Handle NaNs
        frag_start = int(frag_start) if not pd.isna(frag_start) else None
        frag_end = int(frag_end) if not pd.isna(frag_end) else None

        pdb_file = f"{pdb_name}.pdb"
        input_path = os.path.join(input_folder, pdb_file)

        if not os.path.exists(input_path):
            print(f"File not found: {pdb_file}")
            continue

        try:
            print(f"Processing: {pdb_file}, Chain: {chain_id}, Fragment: {frag_start}-{frag_end}")

            with open(input_path, 'r') as f:
                lines = f.readlines()

            # Extract frames of the specified chain and fragment
            frames = extract_chain_from_pdb_lines(lines, chain_id, frag_start, frag_end)

            if not frames:
                print(f"  Chain {chain_id} not found or no frames extracted.")
                continue

            for i, frame_lines in enumerate(frames):
                if frag_start is not None:
                    out_name = f"{pdb_name}_chain_{chain_id}_frag_{str(frag_start)}_frame_{i+1}.pdb"
                else: 
                    out_name = f"{pdb_name}_chain_{chain_id}_frag_0_frame_{i+1}.pdb"
                out_path = os.path.join(output_folder, out_name)
                with open(out_path, 'w') as out_f:
                    out_f.writelines(frame_lines)
                print(f"  Saved: {out_name}")
                saved_files.append(out_path)

        except Exception as e:
            print(f"  Failed to process {pdb_file}: {e}")

    # # === DELETE INTERMEDIATE FILES ===
    # for file_path in saved_files:
    #     try:
    #         os.remove(file_path)
    #         print(f"Deleted: {file_path}")
    #     except Exception as e:
    #         print(f"Failed to delete {file_path}: {e}")

# === USER SETTINGS ===
input_folder = f"{work_dir_path}\pdbs"       # Change this
output_folder = f"{work_dir_path}\pdbs_proc_3"     # Change this
dataframe = df_kinetics[["PDB_wild", "MUTATED_CHAIN", "Fragment Start", "Fragment End"]]  # Include fragment columns

# === RUN ===
process_pdbs_from_dataframe_simple(input_folder, output_folder, dataframe)


Processing: 1azu.pdb, Chain: A, Fragment: None-None
  Saved: 1azu_chain_A_frag_0_frame_1.pdb
Processing: 1ba5.pdb, Chain: A, Fragment: None-None
  Saved: 1ba5_chain_A_frag_0_frame_1.pdb
  Saved: 1ba5_chain_A_frag_0_frame_2.pdb
  Saved: 1ba5_chain_A_frag_0_frame_3.pdb
  Saved: 1ba5_chain_A_frag_0_frame_4.pdb
  Saved: 1ba5_chain_A_frag_0_frame_5.pdb
  Saved: 1ba5_chain_A_frag_0_frame_6.pdb
  Saved: 1ba5_chain_A_frag_0_frame_7.pdb
  Saved: 1ba5_chain_A_frag_0_frame_8.pdb
  Saved: 1ba5_chain_A_frag_0_frame_9.pdb
  Saved: 1ba5_chain_A_frag_0_frame_10.pdb
  Saved: 1ba5_chain_A_frag_0_frame_11.pdb
  Saved: 1ba5_chain_A_frag_0_frame_12.pdb
  Saved: 1ba5_chain_A_frag_0_frame_13.pdb
  Saved: 1ba5_chain_A_frag_0_frame_14.pdb
  Saved: 1ba5_chain_A_frag_0_frame_15.pdb
  Saved: 1ba5_chain_A_frag_0_frame_16.pdb
  Saved: 1ba5_chain_A_frag_0_frame_17.pdb
  Saved: 1ba5_chain_A_frag_0_frame_18.pdb
Processing: 1bdc.pdb, Chain: A, Fragment: None-None
  Saved: 1bdc_chain_A_frag_0_frame_1.pdb
  Saved: 1bdc_c

In [7]:
CO_list_vasily = []
structure_names_list = []
pdb_list = []
L_list = []

for pdb in os.listdir('pdbs_proc_3'):
    pdb_path = "pdbs_proc_3/{}".format(pdb)
    print(pdb)
    abs_CO, L = abs_contact_order3(pdb_path)
    CO_list_vasily.append(abs_CO)

    # structure_names_list.append("{}_{}_{}".format(pdb.split("_")[0], pdb.split("_")[2], pdb.split("_")[4]))
    structure_names_list.append(pdb[:-4])
    pdb_list.append(pdb.split("_")[0])
    L_list.append(L)

1aps_chain_A_frag_0_frame_1.pdb
Atoms:  775
L:  98
1aps_chain_A_frag_0_frame_2.pdb
Atoms:  775
L:  98
1aps_chain_A_frag_0_frame_3.pdb
Atoms:  775
L:  98
1aps_chain_A_frag_0_frame_4.pdb
Atoms:  775
L:  98
1aps_chain_A_frag_0_frame_5.pdb
Atoms:  775
L:  98
1ayi_chain_A_frag_0_frame_1.pdb
Atoms:  670
L:  86
1azu_chain_A_frag_0_frame_1.pdb
Atoms:  930
L:  126
1ba5_chain_A_frag_0_frame_1.pdb
Atoms:  470
L:  53
1ba5_chain_A_frag_0_frame_10.pdb
Atoms:  470
L:  53
1ba5_chain_A_frag_0_frame_11.pdb
Atoms:  470
L:  53
1ba5_chain_A_frag_0_frame_12.pdb
Atoms:  470
L:  53
1ba5_chain_A_frag_0_frame_13.pdb
Atoms:  470
L:  53
1ba5_chain_A_frag_0_frame_14.pdb
Atoms:  470
L:  53
1ba5_chain_A_frag_0_frame_15.pdb
Atoms:  470
L:  53
1ba5_chain_A_frag_0_frame_16.pdb
Atoms:  470
L:  53
1ba5_chain_A_frag_0_frame_17.pdb
Atoms:  470
L:  53
1ba5_chain_A_frag_0_frame_18.pdb
Atoms:  470
L:  53
1ba5_chain_A_frag_0_frame_2.pdb
Atoms:  470
L:  53
1ba5_chain_A_frag_0_frame_3.pdb
Atoms:  470
L:  53
1ba5_chain_A_frag_0_f

In [12]:
df_co = pd.DataFrame({"PDB": pdb_list,"Structure name": structure_names_list, "CO": CO_list_vasily, "Length, aa": L_list, })
df_co["PDB"].unique()
df_co["CO, %"] = df_co["CO"] / df_co["Length, aa"] * 100

In [82]:
# Merge data frames
df_kinetics = df_kinetics.rename(columns={"PDB_wild": "PDB"})

KPro_df_VA = pd.merge(df_kinetics, df_co, on='PDB', how='inner')

# result = pd.merge(result, df, on='PDB', how='inner')
# result[['log(P)', 'log(S)', 'log(X)']] = np.log(result[['P', 'S', 'X']])

print(result.columns)
print("len: ", len(result))



Index(['PDB', 'MUTATION_PDB', 'MUTATED_CHAIN', 'T', 'ln(kf)_H2O', 'ln(ku)_H2O',
       'K_diff', 'Energy, kcal/mol', 'log(Energy, kcal/mol)', 'Folding Type',
       'Fragment Start', 'Fragment End', 'Structure name', 'CO', 'Length, aa',
       'CO, %'],
      dtype='object')
len:  753


In [ ]:
# Import PCS data and merge with "results"
df_psc = pd.read_csv(f'{work_dir_path}\PSC_VA_v1.csv', index_col=0)
df_psc = df_psc.rename(columns={"protid": "Structure name"})
df_psc = df_psc.rename(columns={"C": "X"})

df_psc.loc[:,'Structure name'] = [name[:-2] for name in df_psc["Structure name"]]

print(df_psc.columns)
KPro_df_VA = pd.merge(df_psc, KPro_df_VA, on='Structure name', how='inner')

KPro_df_VA['log(P)'] = np.log(KPro_df_VA["P"])
KPro_df_VA['log(S)'] = np.log(KPro_df_VA["S"])
KPro_df_VA['log(X)'] = np.log(KPro_df_VA["X"])

KPro_df_VA['P_normalized'] = KPro_df_VA["P"]/(KPro_df_VA["Length, aa"]**1.79*(10**(-0.41)))
KPro_df_VA["S_normalized"] = KPro_df_VA["S"]/(KPro_df_VA["Length, aa"]**2.68*(10**(-2.11)))
KPro_df_VA["X_normalized"] = KPro_df_VA["X"]/(KPro_df_VA["Length, aa"]**1.72*(10**(-0.57)))

# Count occurrences of each PDB value
pdb_counts = KPro_df_VA['PDB'].value_counts()
# Compute weights: inverse of the count
weights = 1 / pdb_counts
# Map the weights back to the DataFrame
KPro_df_VA['weights'] = KPro_df_VA['PDB'].map(weights)

KPro_df_VA.to_csv(f'{work_dir_path}\KPro_result_VA_v1.csv')

KPro_df_VA


Index(['Structure name', 'P', 'S', 'X', 'I', 'T', 'L'], dtype='object')


### Make final DataFrame

In [6]:
KPro_df_VA = pd.read_csv(f'{work_dir_path}\KPro_result_VA_v1.csv')
KPro_df_VA

,Unnamed: 0,Structure name,P,S,X,I,T_x,L,PDB,MUTATION_PDB,...,CO,"Length, aa","CO, %",log(P),log(S),log(X),P_normalized,S_normalized,X_normalized,weights
0,0,1azu_chain_A_frag_0_frame_1,5472,5104,2465,0.420,0.391,0.189,1azu,WT,...,20.855979,126,16.552364,8.607399,8.537780,7.809947,2.446148,1.544961,2.234496,1.000000
1,1,1ba5_chain_A_frag_0_frame_1,612,1857,381,0.215,0.652,0.134,1ba5,WT,...,6.077742,53,11.467438,6.416732,7.526718,5.942799,1.289133,5.724698,1.531689,0.055556
2,2,1ba5_chain_A_frag_0_frame_10,476,1447,288,0.215,0.654,0.130,1ba5,WT,...,6.047486,53,11.410351,6.165418,7.277248,5.662960,1.002659,4.460764,1.157812,0.055556
3,3,1ba5_chain_A_frag_0_frame_11,918,2312,511,0.245,0.618,0.137,1ba5,WT,...,6.903721,53,13.025888,6.822197,7.745868,6.236370,1.933699,7.127357,2.054312,0.055556
4,4,1ba5_chain_A_frag_0_frame_12,865,2174,447,0.248,0.624,0.128,1ba5,WT,...,6.376047,53,12.030277,6.762730,7.684324,6.102559,1.822058,6.701935,1.797020,0.055556
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
748,748,2vxd_chain_A_frag_0_frame_6,817,2742,446,0.204,0.685,0.111,2vxd,WT,...,6.567774,54,12.162545,6.705639,7.916443,6.100319,1.664322,8.039928,1.736271,0.050000
749,749,2vxd_chain_A_frag_0_frame_7,652,2809,455,0.166,0.717,0.116,2vxd,WT,...,6.354660,54,11.767890,6.480045,7.940584,6.120297,1.328198,8.236382,1.771308,0.050000
750,750,2vxd_chain_A_frag_0_frame_8,621,2758,449,0.162,0.720,0.117,2vxd,WT,...,5.886302,54,10.900559,6.431331,7.922261,6.107023,1.265047,8.086843,1.747950,0.050000
751,751,2vxd_chain_A_frag_0_frame_9,880,2963,528,0.201,0.678,0.121,2vxd,WT,...,6.208458,54,11.497145,6.779922,7.993958,6.269096,1.792660,8.687931,2.055496,0.050000


In [7]:
columns_to_average = ["Length, aa",	"CO", "log(P)", "log(S)", "log(X)", "Energy, kcal/mol","log(Energy, kcal/mol)" ,"ln(kf)_H2O" ,"ln(ku)_H2O" ]

KPro_df_VA_avg = KPro_df_VA.groupby('PDB')[columns_to_average].mean().reset_index()
KPro_df_VA_avg["weights"] = 1.0
KPro_df_VA_avg["Length^2/3"] = KPro_df_VA_avg["Length, aa"] ** (2/3)
KPro_df_VA_avg

,PDB,"Length, aa",CO,log(P),log(S),log(X),"Energy, kcal/mol","log(Energy, kcal/mol)",ln(kf)_H2O,ln(ku)_H2O,weights,Length^2/3
0,1azu,126.0,20.855979,8.607399,8.537780,7.809947,12.916172,2.558480,4.840000,-16.960000,1.0,25.133156
1,1ba5,53.0,6.319152,6.542717,7.566193,5.979217,2.814304,1.034715,5.910000,1.160000,1.0,14.109683
2,1bdc,60.0,5.418660,6.531024,7.734267,5.938926,4.610159,1.528262,11.700000,4.220000,1.0,15.326189
3,1c9o,66.0,11.154007,6.994850,7.170120,6.129050,4.544360,1.513887,7.220000,-0.450000,1.0,16.331621
4,1coa,64.0,9.895021,6.916715,7.299797,6.242223,8.635468,2.150518,4.890000,-9.685000,1.0,16.000000
5,1csp,67.0,10.984574,6.854355,6.974479,5.986452,2.381789,0.867852,6.320000,2.300000,1.0,16.496173
6,1div,74.0,9.935344,6.852240,7.782405,6.181871,4.830727,1.544265,5.526667,-2.626667,1.0,17.626029
7,1dkt,72.0,10.645266,7.181592,7.338888,6.146329,4.079407,1.405952,4.520000,-2.730000,1.0,17.306995
8,1e0l,37.0,6.087841,5.792826,5.432187,4.783903,2.374496,0.864785,10.050000,5.830000,1.0,11.103702
9,1fex,59.0,6.132711,6.392488,7.728999,6.005773,3.460112,1.241301,8.730000,2.890000,1.0,15.155421


### Plotings

#### CO

In [271]:
# result.plot.scatter(x = "Relative CO", y = "log(Energy, kcal/mol)")

# result.plot.scatter(x = "log(P)", y = "ln(ku)_H2O")
# fig = plt.figure(figsize=(4, 4))
# fig = px.scatter(result, x="Length, aa", y="ln(kf)_H2O", hover_name = "PDB", width=800, height=400)
# fig.show()

fig = px.scatter(KPro_df_VA, x="CO", y="ln(kf)_H2O", hover_name = "PDB", color="Length, aa", width=800, height=400, opacity = 0.3)
fig.show()
fig = px.scatter(KPro_df_VA, x="CO", y="ln(ku)_H2O", hover_name = "PDB", color="Length, aa", width=800, height=400, opacity = 0.3)
fig.show()
fig = px.scatter(KPro_df_VA, x="CO", y="Energy, kcal/mol", hover_name = "PDB", color="Length, aa", width=800, height=400, opacity = 0.3)
fig.show()


#### CT parameters

In [279]:
fig = make_subplots(rows=3, cols=3)

fig.add_trace(go.Scatter(x=KPro_df_VA_avg["log(P)"], y=KPro_df_VA_avg["log(Energy, kcal/mol)"],mode="markers"), row = 1, col = 1)
fig.add_trace(go.Scatter(x=KPro_df_VA_avg["log(S)"], y=KPro_df_VA_avg["log(Energy, kcal/mol)"],mode="markers"), row = 2, col = 1)
fig.add_trace(go.Scatter(x=KPro_df_VA_avg["log(X)"], y=KPro_df_VA_avg["log(Energy, kcal/mol)"],mode="markers"), row = 3, col = 1)


fig.add_trace(go.Scatter(x=KPro_df_VA_avg["log(P)"], y=KPro_df_VA_avg["ln(kf)_H2O"],mode="markers"), row = 1, col = 2)
fig.add_trace(go.Scatter(x=KPro_df_VA_avg["log(S)"], y=KPro_df_VA_avg["ln(kf)_H2O"],mode="markers"), row = 2, col = 2)
fig.add_trace(go.Scatter(x=KPro_df_VA_avg["log(X)"], y=KPro_df_VA_avg["ln(kf)_H2O"],mode="markers"), row = 3, col = 2)

fig.add_trace(go.Scatter(x=KPro_df_VA_avg["log(P)"], y=KPro_df_VA_avg["ln(ku)_H2O"],mode="markers"), row = 1, col = 3)
fig.add_trace(go.Scatter(x=KPro_df_VA_avg["log(S)"], y=KPro_df_VA_avg["ln(ku)_H2O"],mode="markers"), row = 2, col = 3)
fig.add_trace(go.Scatter(x=KPro_df_VA_avg["log(X)"], y=KPro_df_VA_avg["ln(ku)_H2O"],mode="markers"), row = 3, col = 3)


fig.update_layout(height=800, width=800,)
fig.show()


#### CT from length

In [273]:
fig = px.scatter(KPro_df_VA_VA, x="Length, aa", y="P", hover_name = "PDB", color="Length, aa", width=800, height=400, opacity = 0.3, log_x=True, log_y=True)

a = np.linspace(10,160,5)
b = a**1.79 * (10**(-0.41))
fig.add_scatter(x=a, y=b)
fig.show()


fig = px.scatter(KPro_df_VA, x="Length, aa", y="S", hover_name = "PDB", color="Length, aa", width=800, height=400, opacity = 0.3, log_x=True, log_y=True)

a = np.linspace(10,160,5)
b = a**2.68 * (10**(-2.11))
fig.add_scatter(x=a, y=b)
fig.show()


fig = px.scatter(KPro_df_VA, x="Length, aa", y="X", hover_name = "PDB", color="Length, aa", width=800, height=400, opacity = 0.3, log_x=True, log_y=True)

a = np.linspace(10,160,5)
b = a**1.72 * (10**(-0.57))
fig.add_scatter(x=a, y=b)
fig.show()



# KPro_df_VA['P_normalized'] = KPro_df_VA["P"]/(KPro_df_VA["Length, aa"]**1.79*(10**(-0.41)))
# KPro_df_VA["S_normalized"] = KPro_df_VA["S"]/(KPro_df_VA["Length, aa"]**2.68*(10**(-2.11)))
# KPro_df_VA["X_normalized"] = KPro_df_VA["X"]/(KPro_df_VA["Length, aa"]**1.72*(10**(-0.57)))

NameError: name 'KPro_df_VA_VA' is not defined

#### Normalized CT

In [ ]:
fig = px.scatter(KPro_df_VA, x="Length, aa", y="P_normalized", hover_name = "PDB", color="Length, aa", width=800, height=400, opacity = 0.3)
fig.show()
fig = px.scatter(KPro_df_VA, x="Length, aa", y="S_normalized", hover_name = "PDB", color="Length, aa", width=800, height=400, opacity = 0.3)
fig.show()
fig = px.scatter(KPro_df_VA, x="Length, aa", y="X_normalized", hover_name = "PDB", color="Length, aa", width=800, height=400, opacity = 0.3)
fig.show()

#### Length as predictor

In [ ]:
fig = px.scatter(KPro_df_VA, x="Length, aa", y="ln(kf)_H2O", hover_name = "PDB", color="Length, aa", width=800, height=400, opacity = 0.3)
fig.show()
fig = px.scatter(KPro_df_VA, x="Length, aa", y="ln(ku)_H2O", hover_name = "PDB", color="Length, aa", width=800, height=400, opacity = 0.3)
fig.show()
fig = px.scatter(KPro_df_VA, x="Length, aa", y="Energy, kcal/mol", hover_name = "PDB", color="Length, aa", width=800, height=400, opacity = 0.3)
fig.show()


#### CT devide on lenght

In [ ]:
KPro_df_VA['log(P)/L'] = KPro_df_VA["P"]/KPro_df_VA["Length, aa"]
KPro_df_VA['log(S)/L'] = KPro_df_VA["S"]/KPro_df_VA["Length, aa"]
KPro_df_VA['log(X)/L'] = KPro_df_VA["X"]/KPro_df_VA["Length, aa"]


fig = px.scatter(KPro_df_VA, x="X_normalized", y="ln(kf)_H2O", hover_name = "PDB", color="Length, aa", width=800, height=400, opacity = 0.3)
fig.show()
fig = px.scatter(KPro_df_VA, x="X_normalized", y="ln(ku)_H2O", hover_name = "PDB", color="Length, aa", width=800, height=400, opacity = 0.3)
fig.show()
fig = px.scatter(KPro_df_VA, x="X_normalized", y="Energy, kcal/mol", hover_name = "PDB", color="Length, aa", width=800, height=400, opacity = 0.3)
fig.show()


#### 3D plots, logCT

In [ ]:
fig = px.scatter_3d(KPro_df_VA, 
                    x='log(P)', y='log(S)', z='log(X)',
                    color='ln(ku)_H2O',
                    hover_name = "PDB",
                    width=800, height=400,
                    )
fig.update_traces(marker_size = 3)
fig.show()

#### Energy predictions by CO, CT  and length

In [ ]:
fig = px.scatter(KPro_df_VA, x="log(S)", y="ln(kf)_H2O", hover_name = "PDB", color="Length, aa", width=800, height=400, opacity = 0.3)
fig.show()

fig = px.scatter(KPro_df_VA, x="CO", y="ln(kf)_H2O", hover_name = "PDB", color="Length, aa", width=800, height=400, opacity = 0.3)
fig.show()

fig = px.scatter(KPro_df_VA, x="Length, aa", y="ln(kf)_H2O", hover_name = "PDB", color="Length, aa", width=800, height=400, opacity = 0.3)
fig.show()

## Linear regression model

In [255]:
from sklearn.metrics import mean_squared_error
from scipy import stats

def evaluate_model(train, test, feature_columns, target_column, model_name):

    coefficients = []
    intercepts = []
    coefficient_errors = []
    intercept_errors = []
    

    X_train = train[feature_columns]
    y_train = train[target_column]

    X_test = test[feature_columns]
    y_test = test[target_column]
    
    

    X_train = sm.add_constant(X_train)
    X_test = sm.add_constant(X_test)
    
    # model = sm.WLS(y_train, X_train, weights=weights_train).fit()
    model = sm.OLS(y_train, X_train, ).fit()
    # print(model.params)

    intercepts.append(model.params['const'])
    coefficients.append(model.params.iloc[1:].values)
    coefficient_errors.append(model.bse.iloc[1:].values)
    intercept_errors.append(model.bse.iloc[0])

    avg_coefficients = np.mean(coefficients, axis=0)
    avg_intercept = np.mean(intercepts)
    avg_coefficient_errors = np.mean(coefficient_errors, axis=0)
    avg_intercept_error = np.mean(intercept_errors)
    
    formatted_coefficients = [round(coef, 3) for coef in avg_coefficients]
    formatted_coefficient_errors = [round(err, 3) for err in avg_coefficient_errors]
    
    MSE_train = mean_squared_error(y_train, model.predict(X_train))
    rpearson_train, pvalue_train = stats.pearsonr(y_train, model.predict(X_train))

    MSE_test = mean_squared_error(y_test, model.predict(X_test))
    rpearson_test, pvalue_test = stats.pearsonr(y_test, model.predict(X_test))
    
    result = {
        "Model": model_name,
        "Feature Columns": ", ".join(feature_columns),
        "Target Column": target_column,

        "Coefficients": formatted_coefficients,
        "Coefficient Errors": formatted_coefficient_errors,
        "Intercept": round(avg_intercept, 3),
        "Intercept Error": round(avg_intercept_error, 3),

        "MSE train": round(MSE_train, 3),
        "MSE test": round(MSE_test, 3),

        "R train": round(rpearson_train, 3),
        "R test": round(rpearson_test, 3),

        "p-value train": round(pvalue_train, 3),
        "p-value test": round(pvalue_test, 3),

    }
    return result


models = [
    {"model_name": "length_energy", "features": ['Length, aa'], "target": 'Energy, kcal/mol'},
    {"model_name": "CO_energy", "features": ['CO'], "target": 'Energy, kcal/mol'},
    {"model_name": "logCT_logEnergy", "features": ['log(P)', 'log(S)', 'log(X)'], "target": 'log(Energy, kcal/mol)'},

    {"model_name": "length_logkf", "features": ['Length, aa'], "target": 'ln(kf)_H2O'},
    {"model_name": "CO_logkf", "features": ['CO'], "target": 'ln(kf)_H2O'},
    {"model_name": "logCT_logkf", "features": ['log(P)', 'log(S)', 'log(X)'], "target": 'ln(kf)_H2O'},

    {"model_name": "length_logku", "features": ['Length, aa'], "target": 'ln(ku)_H2O'},
    {"model_name": "CO_logku", "features": ['CO'], "target": 'ln(ku)_H2O'},
    {"model_name": "logCT_logku", "features": ['log(P)', 'log(S)', 'log(X)'], "target": 'ln(ku)_H2O'},
]

results = []

#split data frame on test and train proteins structures
unique_proteins = KPro_df_VA_avg["PDB"].unique()
train_proteins, test_proteins = train_test_split(unique_proteins, test_size=0.5, random_state=3724761)
train = KPro_df_VA_avg[KPro_df_VA_avg["PDB"].isin(train_proteins)]
test = KPro_df_VA_avg[KPro_df_VA_avg["PDB"].isin(test_proteins)]


#train the models
for model in models:
    print(model)
    result = evaluate_model(train,test,
                            feature_columns = model["features"],
                            target_column = model["target"],
                            model_name = model["model_name"])
    results.append(result)

KPro_regression_VA = pd.DataFrame(results)
KPro_regression_VA['params'] = KPro_regression_VA.apply(lambda row: row['Coefficients'] + [row['Intercept']], axis=1)
KPro_regression_VA['name_list'] = [
                'Length, aa vs Energy',
                'Contact Order  vs Energy',
                'Log P, S, X vs logEnergy',

                'Length, aa vs log folding rate',
                'Contact Order vs log folding rate',
                'Log P, S, X vs log folding rate',
                
                'Length, aa vs log unfolding rate',
                'Contact Order  vs log unfolding rate',
                'Log P, S, X vs log unfolding rate',
]
KPro_regression_VA

{'model_name': 'length_energy', 'features': ['Length, aa'], 'target': 'Energy, kcal/mol'}
{'model_name': 'CO_energy', 'features': ['CO'], 'target': 'Energy, kcal/mol'}
{'model_name': 'logCT_logEnergy', 'features': ['log(P)', 'log(S)', 'log(X)'], 'target': 'log(Energy, kcal/mol)'}
{'model_name': 'length_logkf', 'features': ['Length, aa'], 'target': 'ln(kf)_H2O'}
{'model_name': 'CO_logkf', 'features': ['CO'], 'target': 'ln(kf)_H2O'}
{'model_name': 'logCT_logkf', 'features': ['log(P)', 'log(S)', 'log(X)'], 'target': 'ln(kf)_H2O'}
{'model_name': 'length_logku', 'features': ['Length, aa'], 'target': 'ln(ku)_H2O'}
{'model_name': 'CO_logku', 'features': ['CO'], 'target': 'ln(ku)_H2O'}
{'model_name': 'logCT_logku', 'features': ['log(P)', 'log(S)', 'log(X)'], 'target': 'ln(ku)_H2O'}


,Model,Feature Columns,Target Column,Coefficients,Coefficient Errors,Intercept,Intercept Error,MSE train,MSE test,R train,R test,p-value train,p-value test,params,name_list
0,length_energy,"Length, aa","Energy, kcal/mol",[0.083],[0.017],-1.475,1.381,3.390,2.109,0.706,0.762,0.000,0.000,"[0.083, -1.475]","Length, aa vs Energy"
1,CO_energy,CO,"Energy, kcal/mol",[0.399],[0.097],0.498,1.142,3.903,3.195,0.650,0.553,0.000,0.003,"[0.399, 0.498]",Contact Order vs Energy
2,logCT_logEnergy,"log(P), log(S), log(X)","log(Energy, kcal/mol)","[0.342, 0.089, 0.139]","[0.324, 0.124, 0.408]",-2.604,0.703,0.128,0.105,0.784,0.742,0.000,0.000,"[0.342, 0.089, 0.139, -2.604]","Log P, S, X vs logEnergy"
3,length_logkf,"Length, aa",ln(kf)_H2O,[-0.056],[0.026],9.926,2.057,7.524,12.824,0.411,0.346,0.041,0.083,"[-0.056, 9.926]","Length, aa vs log folding rate"
4,CO_logkf,CO,ln(kf)_H2O,[-0.45],[0.115],10.576,1.346,5.426,7.924,0.633,0.719,0.001,0.000,"[-0.45, 10.576]",Contact Order vs log folding rate
5,logCT_logkf,"log(P), log(S), log(X)",ln(kf)_H2O,"[-2.752, 1.722, -0.72]","[2.151, 0.825, 2.712]",16.835,4.676,5.678,7.497,0.611,0.766,0.001,0.000,"[-2.752, 1.722, -0.72, 16.835]","Log P, S, X vs log folding rate"
6,length_logku,"Length, aa",ln(ku)_H2O,[-0.197],[0.035],12.375,2.773,13.670,15.812,0.762,0.720,0.000,0.000,"[-0.197, 12.375]","Length, aa vs log unfolding rate"
7,CO_logku,CO,ln(ku)_H2O,[-1.12],[0.156],9.610,1.835,10.084,8.847,0.831,0.857,0.000,0.000,"[-1.12, 9.61]",Contact Order vs log unfolding rate
8,logCT_logku,"log(P), log(S), log(X)",ln(ku)_H2O,"[-4.128, 2.137, -3.662]","[3.195, 1.224, 4.027]",34.068,6.944,12.519,10.017,0.785,0.865,0.000,0.000,"[-4.128, 2.137, -3.662, 34.068]","Log P, S, X vs log unfolding rate"


The weights of the model are calculated: 1/ error*number of samples(structures)

### Predict dG and prepare df

In [ ]:
def pred_dG(spx_vec, param):
    dG = np.dot(spx_vec, param[:-1]) + param[-1]
    return dG

data = {
    "y_measured_energy": test['Energy, kcal/mol'],
    "y_measured_log10kf": test['ln(kf)_H2O'],
    "y_measured_log10ku": test['ln(ku)_H2O'],

    "y_pred_length_energy": (pred_dG(test[['Length, aa']], param = np.array(KPro_regression_VA.loc[KPro_regression_VA["Model"]=="length_energy",'params'].values[0]))),
    "y_pred_CO_energy": (pred_dG(test[['CO']], param = np.array(KPro_regression_VA.loc[KPro_regression_VA["Model"]=="CO_energy","params"].values[0]))),
    "y_pred_logCT_logEnergy": np.exp(pred_dG(test[['log(P)', 'log(S)', 'log(X)']], param = np.array(KPro_regression_VA.loc[KPro_regression_VA["Model"]=="logCT_logEnergy","params"].values[0]))),

    "y_pred_length_logkf": pred_dG(test[['Length, aa']], param = np.array(KPro_regression_VA.loc[KPro_regression_VA["Model"]=="length_logkf","params"].values[0])),
    "y_pred_CO_logkf": pred_dG(test[['CO']], param = np.array(KPro_regression_VA.loc[KPro_regression_VA["Model"]=="CO_logkf","params"].values[0])),
    "y_pred_logCT_logkf": pred_dG(test[['log(P)', 'log(S)', 'log(X)']], param = np.array(KPro_regression_VA.loc[KPro_regression_VA["Model"]=="logCT_logkf","params"].values[0])),

    "y_pred_length_logku": pred_dG(test[['Length, aa']], param = np.array(KPro_regression_VA.loc[KPro_regression_VA["Model"]=="length_logku","params"].values[0])),
    "y_pred_CO_logku": pred_dG(test[['CO']], param = np.array(KPro_regression_VA.loc[KPro_regression_VA["Model"]=="CO_logku","params"].values[0])),
    "y_pred_logCT_logku": pred_dG(test[['log(P)', 'log(S)', 'log(X)']], param = np.array(KPro_regression_VA.loc[KPro_regression_VA["Model"]=="logCT_logku","params"].values[0])),
}

#Take the mean of values for structures of a protein
df_lineair_regression = pd.DataFrame(data)
df_lineair_regression['PDB'] = test['PDB']
df_lineair_regression

,y_measured_energy,y_measured_log10kf,y_measured_log10ku,y_pred_length_energy,y_pred_CO_energy,y_pred_logCT_logEnergy,y_pred_length_logkf,y_pred_CO_logkf,y_pred_logCT_logkf,y_pred_length_logku,...,y_measured_log10ku_std,y_pred_length_energy_std,y_pred_CO_energy_std,y_pred_logCT_logEnergy_std,y_pred_length_logkf_std,y_pred_CO_logkf_std,y_pred_logCT_logkf_std,y_pred_length_logku_std,y_pred_CO_logku_std,y_pred_logCT_logku_std
0,4.610159,11.700000,4.220000,3.5050,2.660045,3.137639,6.566,8.137603,7.904003,0.5550,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,4.830727,5.526667,-2.626667,4.6670,4.462202,3.637805,5.782,6.105095,6.927989,-2.2030,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,4.079407,4.520000,-2.730000,4.5010,4.745461,3.894656,5.894,5.785630,5.283467,-1.8090,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2.374496,10.050000,5.830000,1.5960,2.927049,1.691429,7.854,7.836472,6.802959,5.0860,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,5.640457,1.240000,-8.280000,7.4060,8.047712,7.963526,3.934,2.061287,2.607375,-8.7040,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,6.138144,6.340000,-4.020000,4.0030,5.104522,4.310192,6.230,5.380674,4.796885,-0.6270,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,1.007224,8.400000,6.700000,1.8450,3.353306,2.163726,7.686,7.355730,5.656144,4.4950,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,3.074997,2.470000,-2.720000,3.3390,5.000959,3.984129,6.678,5.497475,4.201065,0.9490,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,11.547532,7.440000,-12.050000,11.0580,8.127617,9.802235,1.470,1.971169,3.006242,-17.3720,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,4.170181,-0.940000,-8.050000,5.7460,6.847555,5.093954,5.054,3.414848,2.514439,-4.7640,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [269]:
def pred_dG(spx_vec, param):
    dG = np.dot(spx_vec, param[:-1]) + param[-1]
    return dG

data = {
    "y_measured_energy": train['Energy, kcal/mol'],
    "y_measured_log10kf": train['ln(kf)_H2O'],
    "y_measured_log10ku": train['ln(ku)_H2O'],

    "y_pred_length_energy": (pred_dG(train[['Length, aa']], param = np.array(KPro_regression_VA.loc[KPro_regression_VA["Model"]=="length_energy",'params'].values[0]))),
    "y_pred_CO_energy": (pred_dG(train[['CO']], param = np.array(KPro_regression_VA.loc[KPro_regression_VA["Model"]=="CO_energy","params"].values[0]))),
    "y_pred_logCT_logEnergy": np.exp(pred_dG(train[['log(P)', 'log(S)', 'log(X)']], param = np.array(KPro_regression_VA.loc[KPro_regression_VA["Model"]=="logCT_logEnergy","params"].values[0]))),

    "y_pred_length_logkf": pred_dG(train[['Length, aa']], param = np.array(KPro_regression_VA.loc[KPro_regression_VA["Model"]=="length_logkf","params"].values[0])),
    "y_pred_CO_logkf": pred_dG(train[['CO']], param = np.array(KPro_regression_VA.loc[KPro_regression_VA["Model"]=="CO_logkf","params"].values[0])),
    "y_pred_logCT_logkf": pred_dG(train[['log(P)', 'log(S)', 'log(X)']], param = np.array(KPro_regression_VA.loc[KPro_regression_VA["Model"]=="logCT_logkf","params"].values[0])),

    "y_pred_length_logku": pred_dG(train[['Length, aa']], param = np.array(KPro_regression_VA.loc[KPro_regression_VA["Model"]=="length_logku","params"].values[0])),
    "y_pred_CO_logku": pred_dG(train[['CO']], param = np.array(KPro_regression_VA.loc[KPro_regression_VA["Model"]=="CO_logku","params"].values[0])),
    "y_pred_logCT_logku": pred_dG(train[['log(P)', 'log(S)', 'log(X)']], param = np.array(KPro_regression_VA.loc[KPro_regression_VA["Model"]=="logCT_logku","params"].values[0])),
}

#Take the mean of values for structures of a protein
df_lineair_regression = pd.DataFrame(data)
df_lineair_regression['PDB'] = train['PDB']
std_df = df_lineair_regression.groupby('PDB').std().add_suffix('_std')
df_lineair_regression = df_lineair_regression.merge(std_df, on='PDB', how='left')
df_lineair_regr = df_lineair_regression.groupby("PDB").mean()
df_lineair_regression

,y_measured_energy,y_measured_log10kf,y_measured_log10ku,y_pred_length_energy,y_pred_CO_energy,y_pred_logCT_logEnergy,y_pred_length_logkf,y_pred_CO_logkf,y_pred_logCT_logkf,y_pred_length_logku,...,y_measured_log10ku_std,y_pred_length_energy_std,y_pred_CO_energy_std,y_pred_logCT_logEnergy_std,y_pred_length_logkf_std,y_pred_CO_logkf_std,y_pred_logCT_logkf_std,y_pred_length_logku_std,y_pred_CO_logku_std,y_pred_logCT_logku_std
0,12.916172,4.84,-16.960,8.983,8.819535,8.892210,2.870,1.190810,2.226332,-12.447,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2.814304,5.91,1.160,2.924,3.019341,3.120869,6.958,7.732382,7.553392,1.934,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,4.544360,7.22,-0.450,4.003,4.948449,3.590593,6.230,5.556697,5.519203,-0.627,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,8.635468,4.89,-9.685,3.837,4.446114,3.592569,6.342,6.123240,5.876051,-0.233,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2.381789,6.32,2.300,4.086,4.880845,3.297072,6.174,5.632942,5.671624,-0.824,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,3.460112,8.73,2.890,3.422,2.944952,3.018966,6.622,7.816280,8.228053,0.752,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,5.963636,7.21,-3.145,5.663,4.640839,6.030047,5.110,5.903625,5.249989,-4.567,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,0.629706,12.39,11.320,0.185,2.047026,0.892072,8.806,8.828978,8.561256,8.435,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,3.495661,2.02,-3.880,6.493,5.915547,6.104705,4.550,4.465984,4.592121,-6.537,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,5.167209,2.70,-6.170,5.580,6.476527,5.764743,5.166,3.833300,4.299917,-4.370,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Use block below to save dataframe

In [236]:
# df_lineair_regr.to_csv('/CT_analysis/Dataframes/KPRO_pred_df_v0')

Load in dataframe

In [237]:
# df_lineair_regr = pd.read_csv('/CT_analysis/Dataframes/KPRO_pred_df_v0', index_col=0)

### Plot free energy of folding predictions

#### Energy Plot

In [239]:
# This code plots the CT parameters and contact order in the same plot
predicted_list = ['y_pred_length_energy', 'y_pred_CO_energy','y_pred_logCT_logEnergy']
color_list = ['Length, aa','Contact Order','CT parameters',]
colors = ['blue', 'red', 'green']

experimental_data = 'y_measured_energy'


axis_min = np.min(df_lineair_regression[predicted_list+[experimental_data]]) // 1
axis_max = -((-np.max(df_lineair_regression[predicted_list+[experimental_data]])) // 1 )

axis_range = [axis_min, axis_max]

fig = go.Figure()

for i, (model_data, label, color) in enumerate(zip(predicted_list, color_list, colors)):

    # Linear regression
    slope, intercept, rvalue, _, _ = stats.linregress(df_lineair_regression[experimental_data], df_lineair_regression[model_data])


    fig.add_trace(go.Scatter(
        x=df_lineair_regression[experimental_data],
        y= df_lineair_regression[model_data],
        mode="markers",
        marker_symbol='x',
        marker=dict(color=color, size=10, opacity=0.8,
                    line=dict(color="black", width=1)),
        name=f"{label}",
        # error_x=dict(
        #     type='data',
        #     array=error_x_values,
        #     visible=True,
        #     thickness=1.5,
        #     width=8,
        #     color=color
        # )
    ))

    x_range = np.linspace(axis_range[0],axis_range[1], 20)
    y_pred = slope * x_range + intercept
    fig.add_trace(go.Scatter(
        x=x_range,
        y=y_pred,
        mode="lines",
        line=dict(color=color, width=3),
        showlegend=False,
    ))

#Calculate statistical parameters and add as anotations to the plot
for i, (model_data, label, color) in enumerate(zip(predicted_list, color_list, colors)):
    MSE = mean_squared_error(df_lineair_regression[experimental_data], df_lineair_regression[model_data])
    rpearson, pvalue = stats.pearsonr(df_lineair_regression[experimental_data], df_lineair_regression[model_data])

    #number of symbols in sign
    n_s = len(f"R=0.00 p-value=0.00 MSE=0.00 ")
    n_s1 = len(f"R={rpearson:.2f} ")
    n_s2 = len(f"p-value={pvalue:.2f} ")
    n_s3 = len(f"MSE = {MSE:.2f} ")

    fig.add_annotation(
        x = axis_range[0] + (axis_range[1]-axis_range[0]) * (n_s1 * 0. / n_s),
        y = axis_range[1] - (axis_range[1]-axis_range[0])/14*(i+0.5),
        text=f"R={rpearson:.2f}",
        showarrow=False,
        font=dict(size=32, color=color),
        xanchor="left",
        bgcolor="rgba(255, 255, 255, 0.7)",
    )
    fig.add_annotation(
        x=axis_range[0] + (axis_range[1]-axis_range[0]) * ((n_s1 + n_s2 * 0.) / n_s),
        y=axis_range[1] - (axis_range[1]-axis_range[0])/14*(i+0.5),
        text=f"p-value={pvalue:.2f}",
        showarrow=False,
        font=dict(size=32, color=color),
        xanchor="left",
        bgcolor="rgba(255, 255, 255, 0.7)",
    )    
    fig.add_annotation(
        x=axis_range[0] + (axis_range[1]-axis_range[0]) * ((n_s1 + n_s2 + n_s3 * 0.) / n_s),
        y=axis_range[1] - (axis_range[1]-axis_range[0])/14*(i+0.5),
        text=f"MSE = {MSE:.2f}",
        showarrow=False,
        font=dict(size=32, color=color),
        xanchor="left",
        bgcolor="rgba(255, 255, 255, 0.7)",
    )

#add diagonal line
fig.add_trace(go.Scatter(
    x=axis_range,
    y=axis_range,
    mode='lines',
    line=dict(color='black', width=1),
    showlegend=False
))


#Plot tytle
fig.update_layout(
    title_text=" Predicted free energy",
    title_x=0.5,
    title_font=dict(size=32),
    width=1150,
    height=750,
    showlegend=True,
    plot_bgcolor="white",
    font=dict(family="Arial", size=32, color='black'),
    margin=dict(l=200, r=100, t=80, b=100)
)

#Set axis names and ranges
fig.update_xaxes(
    title_text="∆𝐺<sub>exp</sub>(kcal/mol)",
    range=axis_range,
    dtick=1,
    title_font=dict(size=32),
    tickfont=dict(size=32),
    showline=True,
    linecolor="black",
    mirror=True,
)

fig.update_yaxes(
    title_text="∆𝐺<sub>pred</sub>(kcal/mol)",
    range=axis_range,
    dtick=1,
    title_font=dict(size=32),
    tickfont=dict(size=32),
    showline=True,
    linecolor="black",
    mirror=True,
)

fig.write_image(f'{work_dir_path}/images/models_energy_v0.png', scale=1)
fig.show()

#### Folding Rate Plot

In [195]:
from sklearn.metrics import mean_squared_error
from scipy import stats


# This code plots the CT parameters and contact order in the same plot
predicted_list = ['y_pred_length_logkf', 'y_pred_CO_logkf','y_pred_logCT_logkf']
color_list = ['Length, aa', 'Contact Order','CT parameters']
colors = ['blue', 'red', 'green']

experimental_data = 'y_measured_log10kf'

axis_min = np.min(df_lineair_regression[predicted_list+[experimental_data]]) // 1
axis_max = -((-np.max(df_lineair_regression[predicted_list+[experimental_data]])) // 1 )
axis_range = [axis_min, axis_max]

fig = go.Figure()

for i, (model_data, label, color) in enumerate(zip(predicted_list, color_list, colors)):

    # Linear regression
    slope, intercept, rvalue, _, _ = stats.linregress(df_lineair_regression[experimental_data], df_lineair_regression[model_data])


    fig.add_trace(go.Scatter(
        x=df_lineair_regression[experimental_data],
        y= df_lineair_regression[model_data],
        mode="markers",
        marker_symbol='x',
        marker=dict(color=color, size=10, opacity=0.8,
                    line=dict(color="black", width=1)),
        name=f"{label}",
        # error_x=dict(
        #     type='data',
        #     array=error_x_values,
        #     visible=True,
        #     thickness=1.5,
        #     width=8,
        #     color=color
        # )
    ))

    x_range = np.linspace(axis_range[0],axis_range[1], 2)
    y_pred = slope * x_range + intercept
    fig.add_trace(go.Scatter(
        x=x_range,
        y=y_pred,
        mode="lines",
        line=dict(color=color, width=3),
        showlegend=False,
    ))

#Calculate statistical parameters and add as anotations to the plot
for i, (model_data, label, color) in enumerate(zip(predicted_list, color_list, colors)):
    MSE = mean_squared_error(df_lineair_regression[experimental_data], df_lineair_regression[model_data])
    rpearson, pvalue = stats.pearsonr(df_lineair_regression[experimental_data], df_lineair_regression[model_data])

    #number of symbols in sign
    n_s = len(f"R=0.00 p-value=0.00 MSE=0.00 ")
    n_s1 = len(f"R={rpearson:.2f} ")
    n_s2 = len(f"p-value={pvalue:.2f} ")
    n_s3 = len(f"MSE = {MSE:.2f} ")

    fig.add_annotation(
        x = axis_range[0] + (axis_range[1]-axis_range[0]) * (n_s1 * 0. / n_s),
        y = axis_range[1] - (axis_range[1]-axis_range[0])/14*(i+0.5),
        text=f"R={rpearson:.2f}",
        showarrow=False,
        font=dict(size=32, color=color),
        xanchor="left",
        bgcolor="rgba(255, 255, 255, 0.7)",
    )
    fig.add_annotation(
        x=axis_range[0] + (axis_range[1]-axis_range[0]) * ((n_s1 + n_s2 * 0.) / n_s),
        y=axis_range[1] - (axis_range[1]-axis_range[0])/14*(i+0.5),
        text=f"p-value={pvalue:.2f}",
        showarrow=False,
        font=dict(size=32, color=color),
        xanchor="left",
        bgcolor="rgba(255, 255, 255, 0.7)",
    )    
    fig.add_annotation(
        x=axis_range[0] + (axis_range[1]-axis_range[0]) * ((n_s1 + n_s2 + n_s3 * 0.) / n_s),
        y=axis_range[1] - (axis_range[1]-axis_range[0])/14*(i+0.5),
        text=f"MSE = {MSE:.2f}",
        showarrow=False,
        font=dict(size=32, color=color),
        xanchor="left",
        bgcolor="rgba(255, 255, 255, 0.7)",
    )
    
#add diagonal line
fig.add_trace(go.Scatter(
    x=axis_range,
    y=axis_range,
    mode='lines',
    line=dict(color='black', width=1),
    showlegend=False
))


#Plot tytle
fig.update_layout(
    title_text=" Predicted Folding Rate",
    title_x=0.5,
    title_font=dict(size=32),
    width=1150,
    height=750,
    showlegend=True,
    plot_bgcolor="white",
    font=dict(family="Arial", size=32, color='black'),
    margin=dict(l=200, r=100, t=80, b=100)
)

#Set axis names and ranges
fig.update_xaxes(
    title_text="ln k<sub>f,exp</sub> (s<sup>−1</sup>)",
    range=axis_range,
    dtick=1,
    title_font=dict(size=32),
    tickfont=dict(size=32),
    showline=True,
    linecolor="black",
    mirror=True,
)

fig.update_yaxes(
    title_text="ln k<sub>f,pred</sub> (s<sup>−1</sup>)",
    range=axis_range,
    dtick=1,
    title_font=dict(size=32),
    tickfont=dict(size=32),
    showline=True,
    linecolor="black",
    mirror=True,
)


fig.write_image(f'{work_dir_path}/images/models_logkf_v0.png', scale=5)

fig.show()

#### Unfolding rate plot

In [212]:
from sklearn.metrics import mean_squared_error
from scipy import stats


# This code plots the CT parameters and contact order in the same plot
predicted_list = ['y_pred_length_logku', 'y_pred_CO_logku','y_pred_logCT_logku']
color_list = ['Length, aa', 'Contact Order','CT parameters']
colors = ['blue', 'red', 'green']

experimental_data = 'y_measured_log10ku'

axis_min = np.min(df_lineair_regression[predicted_list+[experimental_data]]) // 1
axis_max = -((-np.max(df_lineair_regression[predicted_list+[experimental_data]])) // 1 )
axis_range = [axis_min, axis_max]

fig = go.Figure()

for i, (model_data, label, color) in enumerate(zip(predicted_list, color_list, colors)):

    # Linear regression
    slope, intercept, rvalue, _, _ = stats.linregress(df_lineair_regression[experimental_data], df_lineair_regression[model_data])


    fig.add_trace(go.Scatter(
        x=df_lineair_regression[experimental_data],
        y= df_lineair_regression[model_data],
        mode="markers",
        marker_symbol='x',
        marker=dict(color=color, size=10, opacity=0.8,
                    line=dict(color="black", width=1)),
        name=f"{label}",
        # error_x=dict(
        #     type='data',
        #     array=error_x_values,
        #     visible=True,
        #     thickness=1.5,
        #     width=8,
        #     color=color
        # )
    ))

    x_range = np.linspace(axis_range[0],axis_range[1], 2)
    y_pred = slope * x_range + intercept
    fig.add_trace(go.Scatter(
        x=x_range,
        y=y_pred,
        mode="lines",
        line=dict(color=color, width=3),
        showlegend=False,
    ))

#Calculate statistical parameters and add as anotations to the plot
for i, (model_data, label, color) in enumerate(zip(predicted_list, color_list, colors)):
    MSE = mean_squared_error(df_lineair_regression[experimental_data], df_lineair_regression[model_data])
    rpearson, pvalue = stats.pearsonr(df_lineair_regression[experimental_data], df_lineair_regression[model_data])

    #number of symbols in sign
    n_s = len(f"R=0.00 p-value=0.00 MSE=0.00 ")
    n_s1 = len(f"R={rpearson:.2f} ")
    n_s2 = len(f"p-value={pvalue:.2f} ")
    n_s3 = len(f"MSE = {MSE:.2f} ")

    fig.add_annotation(
        x = axis_range[0] + (axis_range[1]-axis_range[0]) * (n_s1 * 0.0 / n_s),
        y = axis_range[1] - (axis_range[1]-axis_range[0])/14*(i+0.5),
        text=f"R={rpearson:.2f}",
        showarrow=False,
        font=dict(size=32, color=color),
        xanchor="left",
        bgcolor="rgba(255, 255, 255, 0.7)",
    )
    fig.add_annotation(
        x=axis_range[0] + (axis_range[1]-axis_range[0]) * ((n_s1 + n_s2 * 0.0) / n_s),
        y=axis_range[1] - (axis_range[1]-axis_range[0])/14*(i+0.5),
        text=f"p-value={pvalue:.2f}",
        showarrow=False,
        font=dict(size=32, color=color),
        xanchor="left",
        bgcolor="rgba(255, 255, 255, 0.7)",
    )    
    fig.add_annotation(
        x=axis_range[0] + (axis_range[1]-axis_range[0]) * ((n_s1 + n_s2 + n_s3 * 0.0) / n_s),
        y=axis_range[1] - (axis_range[1]-axis_range[0])/14*(i+0.5),
        text=f"MSE = {MSE:.2f}",
        showarrow=False,
        font=dict(size=32, color=color),
        xanchor="left",
        bgcolor="rgba(255, 255, 255, 0.7)",
    )

#add diagonal line
fig.add_trace(go.Scatter(
    x=axis_range,
    y=axis_range,
    mode='lines',
    line=dict(color='black', width=1),
    showlegend=False
))


#Plot tytle
fig.update_layout(
    title_text=" Predicted Unfolding Rate",
    title_x=0.5,
    title_font=dict(size=32),
    width=1150,
    height=750,
    showlegend=True,
    plot_bgcolor="white",
    font=dict(family="Arial", size=32, color='black'),
    margin=dict(l=200, r=100, t=80, b=100)
)

#Set axis names and ranges
fig.update_xaxes(
    title_text="ln k<sub>u,exp</sub> (s<sup>−1</sup>)",
    range=axis_range,
    dtick=1,
    title_font=dict(size=32),
    tickfont=dict(size=32),
    showline=True,
    linecolor="black",
    mirror=True,
)

fig.update_yaxes(
    title_text="ln k<sub>u,pred</sub> (s<sup>−1</sup>)",
    range=axis_range,
    dtick=1,
    title_font=dict(size=32),
    tickfont=dict(size=32),
    showline=True,
    linecolor="black",
    mirror=True,
)

fig.show()

fig.write_image(f'{work_dir_path}/images/models_logku_v0.png', scale=1)

#### Plot all 3

In [ ]:

fig = make_subplots(rows=1, cols=3, )
font_size = 20
color_list = ['Length, aa','Contact Order','CT parameters',]
colors = ['blue', 'red', 'green']

######################################## Plot energy #########################################
# This code plots the CT parameters and contact order in the same plot
predicted_list = ['y_pred_length_energy', 'y_pred_CO_energy','y_pred_logCT_logEnergy']
experimental_data = 'y_measured_energy'



axis_min = np.min(df_lineair_regression[predicted_list+[experimental_data]]) // 1
axis_max = -((-np.max(df_lineair_regression[predicted_list+[experimental_data]])) // 1 )

axis_range = [axis_min, axis_max]



for i, (model_data, label, color) in enumerate(zip(predicted_list, color_list, colors)):

    # Linear regression
    slope, intercept, rvalue, _, _ = stats.linregress(df_lineair_regression[experimental_data], df_lineair_regression[model_data])


    fig.add_trace(go.Scatter(
        x=df_lineair_regression[experimental_data],
        y= df_lineair_regression[model_data],
        mode="markers",
        marker_symbol='x',
        marker=dict(color=color, size=10, opacity=0.8,
                    line=dict(color="black", width=1)),
        name=f"{label}",
        
        ),
        row=1, col=1)

    x_range = np.linspace(axis_range[0],axis_range[1], 20)
    y_pred = slope * x_range + intercept
    fig.add_trace(go.Scatter(
        x=x_range,
        y=y_pred,
        mode="lines",
        line=dict(color=color, width=3),
        showlegend=False,
        
    ),
    row=1, col=1)

#Calculate statistical parameters and add as anotations to the plot
for i, (model_data, label, color) in enumerate(zip(predicted_list, color_list, colors)):
    MSE = mean_squared_error(df_lineair_regression[experimental_data], df_lineair_regression[model_data])
    rpearson, pvalue = stats.pearsonr(df_lineair_regression[experimental_data], df_lineair_regression[model_data])

    #number of symbols in sign
    n_s = len(f"R=0.00 p-value=0.00 MSE=0.00 ")
    n_s1 = len(f"R={rpearson:.2f} ")
    n_s2 = len(f"p-value={pvalue:.2f} ")
    n_s3 = len(f"MSE = {MSE:.2f} ")

    fig.add_annotation(
        x = axis_range[0] + (axis_range[1]-axis_range[0]) * (n_s1 * 0. / n_s),
        y = axis_range[1] - (axis_range[1]-axis_range[0])/14*(i+0.5),
        text=f"R={rpearson:.2f}",
        showarrow=False,
        font=dict(size=font_size, color=color),
        xanchor="left",
        bgcolor="rgba(255, 255, 255, 0.7)",
        row=1, col=1
    )
    fig.add_annotation(
        x=axis_range[0] + (axis_range[1]-axis_range[0]) * ((n_s1 + n_s2 * 0.) / n_s),
        y=axis_range[1] - (axis_range[1]-axis_range[0])/14*(i+0.5),
        text=f"p-value={pvalue:.2f}",
        showarrow=False,
        font=dict(size=font_size, color=color),
        xanchor="left",
        bgcolor="rgba(255, 255, 255, 0.7)",
        row=1, col=1
    )    
    fig.add_annotation(
        x=axis_range[0] + (axis_range[1]-axis_range[0]) * ((n_s1 + n_s2 + n_s3 * 0.) / n_s),
        y=axis_range[1] - (axis_range[1]-axis_range[0])/14*(i+0.5),
        text=f"MSE = {MSE:.2f}",
        showarrow=False,
        font=dict(size=font_size, color=color),
        xanchor="left",
        bgcolor="rgba(255, 255, 255, 0.7)",
        row=1, col=1
    )

#add diagonal line
fig.add_trace(go.Scatter(
    x=axis_range,
    y=axis_range,
    mode='lines',
    line=dict(color='black', width=1),
    showlegend=False,
    ),
    row=1, col=1
    )


#Set axis names and ranges
fig.update_xaxes(
    title_text="∆𝐺<sub>exp</sub>(kcal/mol)",
    range=axis_range,
    dtick=1,
    title_font=dict(size=font_size),
    tickfont=dict(size=font_size),
    showline=True,
    linecolor="black",
    mirror=True,
    row=1, col=1
)

fig.update_yaxes(
    title_text="∆𝐺<sub>pred</sub>(kcal/mol)",
    range=axis_range,
    dtick=1,
    title_font=dict(size=font_size),
    tickfont=dict(size=font_size),
    showline=True,
    linecolor="black",
    mirror=True,
    row=1, col=1
)

################################################################################################################################################################

######################################## Plot kf #########################################
# This code plots the CT parameters and contact order in the same plot
predicted_list = ['y_pred_length_logkf', 'y_pred_CO_logkf','y_pred_logCT_logkf']

experimental_data = 'y_measured_log10kf'

axis_min = np.min(df_lineair_regression[predicted_list+[experimental_data]]) // 1
axis_max = -((-np.max(df_lineair_regression[predicted_list+[experimental_data]])) // 1 )

axis_range = [axis_min, axis_max]

for i, (model_data, label, color) in enumerate(zip(predicted_list, color_list, colors)):

    # Linear regression
    slope, intercept, rvalue, _, _ = stats.linregress(df_lineair_regression[experimental_data], df_lineair_regression[model_data])


    fig.add_trace(go.Scatter(
        x=df_lineair_regression[experimental_data],
        y= df_lineair_regression[model_data],
        mode="markers",
        marker_symbol='x',
        marker=dict(color=color, size=10, opacity=0.8,
                    line=dict(color="black", width=1)),
        name=f"{label}",
        showlegend=False,
        ),
        row=1, col=2)

    x_range = np.linspace(axis_range[0],axis_range[1], 20)
    y_pred = slope * x_range + intercept
    fig.add_trace(go.Scatter(
        x=x_range,
        y=y_pred,
        mode="lines",
        line=dict(color=color, width=3),
        showlegend=False,
        
    ),
    row=1, col=2)

#Calculate statistical parameters and add as anotations to the plot
for i, (model_data, label, color) in enumerate(zip(predicted_list, color_list, colors)):
    MSE = mean_squared_error(df_lineair_regression[experimental_data], df_lineair_regression[model_data])
    rpearson, pvalue = stats.pearsonr(df_lineair_regression[experimental_data], df_lineair_regression[model_data])

    #number of symbols in sign
    n_s = len(f"R=0.00 p-value=0.00 MSE=0.00 ")
    n_s1 = len(f"R={rpearson:.2f} ")
    n_s2 = len(f"p-value={pvalue:.2f} ")
    n_s3 = len(f"MSE = {MSE:.2f} ")

    fig.add_annotation(
        x = axis_range[0] + (axis_range[1]-axis_range[0]) * (n_s1 * 0. / n_s),
        y = axis_range[1] - (axis_range[1]-axis_range[0])/14*(i+0.5),
        text=f"R={rpearson:.2f}",
        showarrow=False,
        font=dict(size=font_size, color=color),
        xanchor="left",
        bgcolor="rgba(255, 255, 255, 0.7)",
        row=1, col=2
    )
    fig.add_annotation(
        x=axis_range[0] + (axis_range[1]-axis_range[0]) * ((n_s1 + n_s2 * 0.) / n_s),
        y=axis_range[1] - (axis_range[1]-axis_range[0])/14*(i+0.5),
        text=f"p-value={pvalue:.2f}",
        showarrow=False,
        font=dict(size=font_size, color=color),
        xanchor="left",
        bgcolor="rgba(255, 255, 255, 0.7)",
        row=1, col=2
    )    
    fig.add_annotation(
        x=axis_range[0] + (axis_range[1]-axis_range[0]) * ((n_s1 + n_s2 + n_s3 * 0.) / n_s),
        y=axis_range[1] - (axis_range[1]-axis_range[0])/14*(i+0.5),
        text=f"MSE = {MSE:.2f}",
        showarrow=False,
        font=dict(size=font_size, color=color),
        xanchor="left",
        bgcolor="rgba(255, 255, 255, 0.7)",
        row=1, col=2
    )

#add diagonal line
fig.add_trace(go.Scatter(
    x=axis_range,
    y=axis_range,
    mode='lines',
    line=dict(color='black', width=1),
    showlegend=False,
    ),
    row=1, col=2
    )


#Set axis names and ranges
fig.update_xaxes(
    title_text="ln k<sub>f,exp</sub>(kcal/mol)",
    range=axis_range,
    dtick=1,
    title_font=dict(size=font_size),
    tickfont=dict(size=font_size),
    showline=True,
    linecolor="black",
    mirror=True,
    row=1, col=2
)

fig.update_yaxes(
    title_text="ln k<sub>f,pred</sub>(kcal/mol)",
    range=axis_range,
    dtick=1,
    title_font=dict(size=font_size),
    tickfont=dict(size=font_size),
    showline=True,
    linecolor="black",
    mirror=True,
    row=1, col=2
)

################################################################################################################################################################

######################################## Plot ku #########################################
# This code plots the CT parameters and contact order in the same plot
predicted_list = ['y_pred_length_logku', 'y_pred_CO_logku','y_pred_logCT_logku']

experimental_data = 'y_measured_log10ku'

axis_min = np.min(df_lineair_regression[predicted_list+[experimental_data]]) // 1
axis_max = -((-np.max(df_lineair_regression[predicted_list+[experimental_data]])) // 1 )

axis_range = [axis_min, axis_max]

for i, (model_data, label, color) in enumerate(zip(predicted_list, color_list, colors)):

    # Linear regression
    slope, intercept, rvalue, _, _ = stats.linregress(df_lineair_regression[experimental_data], df_lineair_regression[model_data])


    fig.add_trace(go.Scatter(
        x=df_lineair_regression[experimental_data],
        y= df_lineair_regression[model_data],
        mode="markers",
        marker_symbol='x',
        marker=dict(color=color, size=10, opacity=0.8,
                    line=dict(color="black", width=1)),
        name=f"{label}",
        showlegend=False,
        ),
        row=1, col=3)

    x_range = np.linspace(axis_range[0],axis_range[1], 20)
    y_pred = slope * x_range + intercept
    fig.add_trace(go.Scatter(
        x=x_range,
        y=y_pred,
        mode="lines",
        line=dict(color=color, width=3),
        showlegend=False,
        
    ),
    row=1, col=3)

#Calculate statistical parameters and add as anotations to the plot
for i, (model_data, label, color) in enumerate(zip(predicted_list, color_list, colors)):
    MSE = mean_squared_error(df_lineair_regression[experimental_data], df_lineair_regression[model_data])
    rpearson, pvalue = stats.pearsonr(df_lineair_regression[experimental_data], df_lineair_regression[model_data])

    #number of symbols in sign
    n_s = len(f"R=0.00 p-value=0.00 MSE=0.00 ")
    n_s1 = len(f"R={rpearson:.2f} ")
    n_s2 = len(f"p-value={pvalue:.2f} ")
    n_s3 = len(f"MSE = {MSE:.2f} ")

    fig.add_annotation(
        x = axis_range[0] + (axis_range[1]-axis_range[0]) * (n_s1 * 0. / n_s),
        y = axis_range[1] - (axis_range[1]-axis_range[0])/14*(i+0.5),
        text=f"R={rpearson:.2f}",
        showarrow=False,
        font=dict(size=font_size, color=color),
        xanchor="left",
        bgcolor="rgba(255, 255, 255, 0.7)",
        row=1, col=3
    )
    fig.add_annotation(
        x=axis_range[0] + (axis_range[1]-axis_range[0]) * ((n_s1 + n_s2 * 0.) / n_s),
        y=axis_range[1] - (axis_range[1]-axis_range[0])/14*(i+0.5),
        text=f"p-value={pvalue:.2f}",
        showarrow=False,
        font=dict(size=font_size, color=color),
        xanchor="left",
        bgcolor="rgba(255, 255, 255, 0.7)",
        row=1, col=3
    )    
    fig.add_annotation(
        x=axis_range[0] + (axis_range[1]-axis_range[0]) * ((n_s1 + n_s2 + n_s3 * 0.) / n_s),
        y=axis_range[1] - (axis_range[1]-axis_range[0])/14*(i+0.5),
        text=f"MSE = {MSE:.2f}",
        showarrow=False,
        font=dict(size=font_size, color=color),
        xanchor="left",
        bgcolor="rgba(255, 255, 255, 0.7)",
        row=1, col=3
    )

#add diagonal line
fig.add_trace(go.Scatter(
    x=axis_range,
    y=axis_range,
    mode='lines',
    line=dict(color='black', width=1),
    showlegend=False,
    ),
    row=1, col=3
    )




#Set axis names and ranges
fig.update_xaxes(
    title_text="ln k<sub>u,exp</sub>(kcal/mol)",
    range=axis_range,
    dtick=2,
    title_font=dict(size=font_size),
    tickfont=dict(size=font_size),
    showline=True,
    linecolor="black",
    mirror=True,
    row=1, col=3
)

fig.update_yaxes(
    title_text="ln k<sub>u,pred</sub>(kcal/mol)",
    range=axis_range,
    dtick=2,
    title_font=dict(size=font_size),
    tickfont=dict(size=font_size),
    showline=True,
    linecolor="black",
    mirror=True,
    row=1, col=3
)

################################################################################################################################################################

#Plot tytle
fig.update_layout(
    # title_text=" Predicted free energy",
    # title_x=0.5,
    # title_font=dict(size=font_size),
    width=1800,
    height=600,
    showlegend=True,
    plot_bgcolor="white",
    font=dict(family="Arial", size=font_size, color='black'),
    # margin=dict(l=200, r=100, t=80, b=100),
    # row=1, col=1
)


fig.write_image(f'{work_dir_path}/images/models_all_train_v1.png', scale=5, 
    width=1800,
    height=600,)

fig.show()

## Prediction model validation on ACPRO dataset

In [4]:
# Load in ACPRO dataframe or prepare in forst part of the code
ACPRO_df = pd.read_csv(f'{work_dir_path}\ACPRO_22avg.csv', index_col=0).loc[:, ['Chain', 'Protein Length', 'Contact Order','ln kf','Fragment start', 'Fragment end','Temperature', 'pH']]
ACPRO_df["PDB"] = ACPRO_df.index

ACPRO_df

,Chain,Protein Length,Contact Order,ln kf,Fragment start,Fragment end,Temperature,pH,PDB
PDB Id,,,,,,,,,
1A6N,A,151,14.00990,1.13,NaN,NaN,5.00,6.2,1A6N
1ADW,A,123,15.36850,0.69,NaN,NaN,15.00,7.0,1ADW
1AON,A,186,43.26270,0.18,191.0,376.0,15.00,7.0,1AON
1APS,A,98,20.72830,-1.58,NaN,NaN,25.00,7.0,1APS
1ARR,A,53,2.71281,9.20,NaN,NaN,25.00,7.5,1ARR
...,...,...,...,...,...,...,...,...,...
2VKN,A,70,13.46580,2.11,NaN,NaN,25.00,7.0,2VKN
2WXC,A,47,4.50322,11.20,NaN,NaN,9.85,7.0,2WXC
3CHY,A,128,11.31200,1.00,NaN,NaN,25.00,7.0,3CHY


### Dowload and pre-process PDBs

In [376]:
import urllib
import gzip
import shutil

for pdb in ACPRO_df.index:
    if not os.path.isfile('pdbs/{}.pdb'.format(pdb.lower())):
        if not os.path.isfile('pdbs_ACPRO/{}.pdb'.format(pdb.lower())):

            try:
                urllib.request.urlretrieve(f"https://files.wwpdb.org/pub/pdb/data/structures/divided/pdb/{pdb.lower()[1:3]}/pdb{pdb.lower()}.ent.gz", f'pdbs_ACPRO/{pdb.lower()}.ent.gz')

                with gzip.open(f'pdbs_ACPRO/{pdb.lower()}.ent.gz', 'rb') as f_in:
                    with open(f'pdbs_ACPRO/{pdb.lower()}.pdb', 'wb') as f_out:
                        shutil.copyfileobj(f_in, f_out)

            except:
                print(f"Problems with {pdb}")
                print(NameError)
            

In [391]:
def is_hydrogen(atom_name):
    return atom_name.startswith('H')

def extract_chain_from_pdb_lines(lines, target_chain_id, frag_start=None, frag_end=None):
    """
    Extracts lines for a specific chain (excluding hydrogens) from a list of PDB lines.
    If frag_start and frag_end are provided, only extracts residues within that range.
    Returns a list of frames (each frame is a list of lines).
    """
    frames = []
    current_frame = []
    inside_model = False

    for line in lines:
        record = line[:6].strip()

        if record == 'MODEL':
            inside_model = True
            current_frame = []
        elif record == 'ENDMDL':
            inside_model = False
            frames.append(current_frame)
        elif record == 'ATOM':
            chain_id = line[21].strip()
            atom_name = line[12:16].strip()
            res_seq = int(line[22:26].strip())

            if chain_id == target_chain_id and not is_hydrogen(atom_name):
                if frag_start is not None and frag_end is not None:
                    if not (frag_start <= res_seq <= frag_end):
                        continue  # Skip residues outside fragment range

                if not inside_model:
                    current_frame = []  # Start frame on-the-fly if no MODEL/ENDMDL
                    inside_model = True

                current_frame.append(line)

        elif record in ('TER', 'END') and inside_model:
            current_frame.append(line)

    if inside_model and current_frame:
        frames.append(current_frame)

    return frames

def process_pdbs_from_dataframe_simple(input_folder, output_folder, dataframe):
    os.makedirs(output_folder, exist_ok=True)
    df = dataframe.copy()

    df['PDB'] = df['PDB'].str.lower().str.strip()
    df['Chain'] = df['Chain'].str.strip()

    saved_files = []  # Track intermediate files

    for _, row in df.iterrows():
        pdb_name = row['PDB']
        chain_id = row['Chain']
        frag_start = row.get("Fragment start", np.nan)
        frag_end = row.get("Fragment end", np.nan)

        # Handle NaNs
        frag_start = int(frag_start) if not pd.isna(frag_start) else None
        frag_end = int(frag_end) if not pd.isna(frag_end) else None

        pdb_file = f"{pdb_name}.pdb"
        input_path = os.path.join(input_folder, pdb_file)

        if not os.path.exists(input_path):
            print(f"File not found: {pdb_file}")
            continue

        try:
            print(f"Processing: {pdb_file}, Chain: {chain_id}, Fragment: {frag_start}-{frag_end}")

            with open(input_path, 'r') as f:
                lines = f.readlines()

            # Extract frames of the specified chain and fragment
            frames = extract_chain_from_pdb_lines(lines, chain_id, frag_start, frag_end)

            if not frames:
                print(f"  Chain {chain_id} not found or no frames extracted.")
                continue

            for i, frame_lines in enumerate(frames):
                if frag_start is not None:
                    out_name = f"{pdb_name}_chain_{chain_id}_frag_{str(frag_start)}_frame_{i+1}.pdb"
                else: 
                    out_name = f"{pdb_name}_chain_{chain_id}_frag_0_frame_{i+1}.pdb"
                out_path = os.path.join(output_folder, out_name)
                with open(out_path, 'w') as out_f:
                    out_f.writelines(frame_lines)
                print(f"  Saved: {out_name}")
                saved_files.append(out_path)

        except Exception as e:
            print(f"  Failed to process {pdb_file}: {e}")

    # # === DELETE INTERMEDIATE FILES ===
    # for file_path in saved_files:
    #     try:
    #         os.remove(file_path)
    #         print(f"Deleted: {file_path}")
    #     except Exception as e:
    #         print(f"Failed to delete {file_path}: {e}")

# === USER SETTINGS ===
input_folder = f"{work_dir_path}\pdbs_ACPRO"       # Change this
output_folder = f"{work_dir_path}\pdbs_ACPRO_proc"     # Change this
dataframe = ACPRO_df[["PDB", "Chain", "Fragment start", "Fragment end"]]  # Include fragment columns

# === RUN ===
process_pdbs_from_dataframe_simple(input_folder, output_folder, dataframe)

Processing: 1a6n.pdb, Chain: A, Fragment: None-None
  Saved: 1a6n_chain_A_frag_0_frame_1.pdb
Processing: 1adw.pdb, Chain: A, Fragment: None-None
  Saved: 1adw_chain_A_frag_0_frame_1.pdb
Processing: 1aon.pdb, Chain: A, Fragment: 191-376
  Saved: 1aon_chain_A_frag_191_frame_1.pdb
File not found: 1aps.pdb
Processing: 1arr.pdb, Chain: A, Fragment: None-None
  Saved: 1arr_chain_A_frag_0_frame_1.pdb
Processing: 1au7.pdb, Chain: A, Fragment: 103-160
  Saved: 1au7_chain_A_frag_103_frame_1.pdb
Processing: 1aue.pdb, Chain: A, Fragment: None-None
  Saved: 1aue_chain_A_frag_0_frame_1.pdb
Processing: 1avz.pdb, Chain: C, Fragment: None-None
  Saved: 1avz_chain_C_frag_0_frame_1.pdb
File not found: 1ayi.pdb
Processing: 1b9c.pdb, Chain: A, Fragment: None-None
  Saved: 1b9c_chain_A_frag_0_frame_1.pdb
File not found: 1ba5.pdb
Processing: 1bd8.pdb, Chain: A, Fragment: None-None
  Saved: 1bd8_chain_A_frag_0_frame_1.pdb
Processing: 1bf4.pdb, Chain: A, Fragment: None-None
  Saved: 1bf4_chain_A_frag_0_frame_1

### Calculate Contact Order

In [392]:
CO_list_vasily = []
structure_names_list = []
pdb_list = []
L_list = []

for pdb in os.listdir('pdbs_ACPRO_proc'):
    pdb_path = "pdbs_ACPRO_proc/{}".format(pdb)
    print(pdb)
    abs_CO, L = abs_contact_order3(pdb_path)
    CO_list_vasily.append(abs_CO)

    # structure_names_list.append("{}_{}_{}".format(pdb.split("_")[0], pdb.split("_")[2], pdb.split("_")[4]))
    structure_names_list.append(pdb[:-4])
    pdb_list.append(pdb.split("_")[0])
    L_list.append(L)

1a6n_chain_A_frag_0_frame_1.pdb
Atoms:  1204
L:  151
1adw_chain_A_frag_0_frame_1.pdb
Atoms:  934
L:  123
1aon_chain_A_frag_191_frame_1.pdb
Atoms:  1362
L:  186
1arr_chain_A_frag_0_frame_1.pdb
Atoms:  435
L:  53
1au7_chain_A_frag_103_frame_1.pdb
Atoms:  482
L:  58
1aue_chain_A_frag_0_frame_1.pdb
Atoms:  781
L:  92
1avz_chain_C_frag_0_frame_1.pdb
Atoms:  463
L:  57
1b9c_chain_A_frag_0_frame_1.pdb
Atoms:  1774
L:  224
1bd8_chain_A_frag_0_frame_1.pdb
Atoms:  1168
L:  156
1bf4_chain_A_frag_0_frame_1.pdb
Atoms:  502
L:  63
1bfe_chain_A_frag_0_frame_1.pdb
Atoms:  826
L:  110
1bni_chain_A_frag_0_frame_1.pdb
Atoms:  851
L:  108
1bnz_chain_A_frag_0_frame_1.pdb
Atoms:  510
L:  64
1brs_chain_D_frag_0_frame_1.pdb
Atoms:  693
L:  87
1cdc_chain_A_frag_0_frame_1.pdb
Atoms:  759
L:  96
1e0g_chain_A_frag_0_frame_1.pdb
Atoms:  382
L:  48
1e0g_chain_A_frag_0_frame_10.pdb
Atoms:  382
L:  48
1e0g_chain_A_frag_0_frame_11.pdb
Atoms:  382
L:  48
1e0g_chain_A_frag_0_frame_12.pdb
Atoms:  382
L:  48
1e0g_chain_A_

In [393]:
df_co = pd.DataFrame({"PDB": pdb_list,"Structure name": structure_names_list, "CO": CO_list_vasily, "Length, aa": L_list, })
df_co["PDB"].unique()
df_co["CO, %"] = df_co["CO"] / df_co["Length, aa"] * 100

### Cast final dataframe

In [436]:

# Merge data frames
ACPRO_df = ACPRO_df.reset_index()
ACPRO_df['PDB'] = ACPRO_df['PDB'].str.lower()
ACPRO_df_VA = pd.merge(ACPRO_df, df_co, on='PDB', how='inner')


# Import PCS data and merge with "results"
df_psc = pd.read_csv(f'{work_dir_path}\ACPRO_PSC_VA_v1.csv', index_col=0)
df_psc = df_psc.rename(columns={"protid": "Structure name"})
df_psc = df_psc.rename(columns={"C": "X"})

df_psc.loc[:,'Structure name'] = [name[:-2] for name in df_psc["Structure name"]]

print(df_psc.columns)
ACPRO_df_VA = pd.merge(df_psc, ACPRO_df_VA, on='Structure name', how='inner')

ACPRO_df_VA['log(P)'] = np.log(ACPRO_df_VA["P"])
ACPRO_df_VA['log(S)'] = np.log(ACPRO_df_VA["S"])
ACPRO_df_VA['log(X)'] = np.log(ACPRO_df_VA["X"])

ACPRO_df_VA['P_normalized'] = ACPRO_df_VA["P"]/(ACPRO_df_VA["Length, aa"]**1.79*(10**(-0.41)))
ACPRO_df_VA["S_normalized"] = ACPRO_df_VA["S"]/(ACPRO_df_VA["Length, aa"]**2.68*(10**(-2.11)))
ACPRO_df_VA["X_normalized"] = ACPRO_df_VA["X"]/(ACPRO_df_VA["Length, aa"]**1.72*(10**(-0.57)))

# Count occurrences of each PDB value
pdb_counts = ACPRO_df_VA['PDB'].value_counts()
# Compute weights: inverse of the count
weights = 1 / pdb_counts
# Map the weights back to the DataFrame
ACPRO_df_VA['weights'] = ACPRO_df_VA['PDB'].map(weights)


ACPRO_df_VA = ACPRO_df_VA.loc[ACPRO_df_VA['P']!=0]
ACPRO_df_VA = ACPRO_df_VA.loc[ACPRO_df_VA['S']!=0]
ACPRO_df_VA = ACPRO_df_VA.loc[ACPRO_df_VA['X']!=0]
ACPRO_df_VA = ACPRO_df_VA[~ACPRO_df_VA["ln kf"].isna()]

ACPRO_df_VA.to_csv(f'{work_dir_path}\ACPRO_result_VA_v1.csv')


ACPRO_df_VA


Index(['Structure name', 'P', 'S', 'X', 'I', 'T', 'L'], dtype='object')


c:\Users\akulovve\Documents\MS_student\Muriel\paper\code\.conda\Lib\site-packages\pandas\core\arraylike.py:399: RuntimeWarning:

divide by zero encountered in log

c:\Users\akulovve\Documents\MS_student\Muriel\paper\code\.conda\Lib\site-packages\pandas\core\arraylike.py:399: RuntimeWarning:

divide by zero encountered in log

c:\Users\akulovve\Documents\MS_student\Muriel\paper\code\.conda\Lib\site-packages\pandas\core\arraylike.py:399: RuntimeWarning:

divide by zero encountered in log



,Structure name,P,S,X,I,T,L,index,PDB Id,Chain,...,CO,"Length, aa","CO, %",log(P),log(S),log(X),P_normalized,S_normalized,X_normalized,weights
0,1a6n_chain_A_frag_0_frame_1,6987,28725,1963,0.185,0.762,0.052,0,1A6N,A,...,12.657346,151,8.382348,8.851807,10.265523,7.582229,2.259030,5.353056,1.303407,1.00
1,1adw_chain_A_frag_0_frame_1,3205,9964,2584,0.203,0.633,0.164,1,1ADW,A,...,15.368533,123,12.494743,8.072467,9.206734,7.857094,1.495884,3.217274,2.441494,1.00
2,1arr_chain_A_frag_0_frame_1,105,1510,215,0.057,0.825,0.117,4,1ARR,A,...,2.712812,53,5.118512,4.653960,7.319865,5.370638,0.221175,4.654978,0.864339,1.00
3,1aue_chain_A_frag_0_frame_1,3184,11999,1288,0.193,0.728,0.078,6,1AUE,A,...,9.874698,92,10.733368,8.065894,9.392579,7.160846,2.499153,8.437109,2.005399,1.00
4,1avz_chain_C_frag_0_frame_1,1109,912,394,0.459,0.378,0.163,7,1AVZ,C,...,11.184179,57,19.621366,7.011214,6.815640,5.976351,2.050763,2.313397,1.397628,1.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
248,2wxc_chain_A_frag_0_frame_8,273,948,210,0.191,0.662,0.147,122,2WXC,A,...,4.672618,47,9.941740,5.609472,6.854355,5.347108,0.713029,4.032609,1.038033,0.05
249,2wxc_chain_A_frag_0_frame_9,268,986,177,0.187,0.689,0.124,122,2WXC,A,...,4.887861,47,10.399704,5.590987,6.893656,5.176150,0.699970,4.194253,0.874913,0.05
250,3chy_chain_A_frag_0_frame_1,2949,14307,1465,0.158,0.764,0.078,123,3CHY,A,...,11.150345,128,8.711207,7.989221,9.568504,7.289611,1.281648,4.151699,1.292518,1.00
251,3f6r_chain_A_frag_0_frame_1,4387,15826,2578,0.192,0.694,0.113,124,3F6R,A,...,15.196951,147,10.338062,8.386401,9.669409,7.854769,1.488228,3.169281,1.792657,1.00


In [5]:
ACPRO_df_VA = pd.read_csv(f'{work_dir_path}\ACPRO_result_VA_v1.csv')

columns_to_average = ["Length, aa","CO", "log(P)", "log(S)", "log(X)", "ln kf"]

ACPRO_df_VA_avg = ACPRO_df_VA.groupby('PDB')[columns_to_average].mean().reset_index()
ACPRO_df_VA_avg["weights"] = 1.0
ACPRO_df_VA_avg["Length^2/3"] = ACPRO_df_VA_avg["Length, aa"] ** (2/3)
ACPRO_df_VA_avg = ACPRO_df_VA_avg.rename(columns={"ln kf": "ln(kf)_H2O"})

ACPRO_df_VA_avg

,PDB,"Length, aa",CO,log(P),log(S),log(X),ln(kf)_H2O,weights,Length^2/3
0,1a6n,151.0,12.657346,8.851807,10.265523,7.582229,1.13,1.0,28.356413
1,1adw,123.0,15.368533,8.072467,9.206734,7.857094,0.69,1.0,24.732617
2,1arr,53.0,2.712812,4.653960,7.319865,5.370638,9.20,1.0,14.109683
3,1aue,92.0,9.874698,8.065894,9.392579,7.160846,5.37,1.0,20.379423
4,1avz,57.0,11.184179,7.011214,6.815640,5.976351,4.88,1.0,14.810961
...,...,...,...,...,...,...,...,...,...
62,2vkn,66.0,13.550933,7.527794,6.566672,6.070738,2.11,1.0,16.331621
63,2wxc,47.0,4.620003,5.404370,6.798646,5.102411,11.20,1.0,13.023626
64,3chy,128.0,11.150345,7.989221,9.568504,7.289611,1.00,1.0,25.398417
65,3f6r,147.0,15.196951,8.386401,9.669409,7.854769,3.52,1.0,27.853400


## Linear regression with combined data ACPRO and K-Pro

In [8]:
# Step 1: Find common columns
common_cols = ACPRO_df_VA_avg.columns.intersection(KPro_df_VA_avg.columns)

# Step 2: Merge on those columns
merged_df_VA = pd.concat([ACPRO_df_VA_avg[common_cols], KPro_df_VA_avg[common_cols]], ignore_index=True)
merged_df_VA


,PDB,"Length, aa",CO,log(P),log(S),log(X),ln(kf)_H2O,weights,Length^2/3
0,1a6n,151.0,12.657346,8.851807,10.265523,7.582229,1.13,1.0,28.356413
1,1adw,123.0,15.368533,8.072467,9.206734,7.857094,0.69,1.0,24.732617
2,1arr,53.0,2.712812,4.653960,7.319865,5.370638,9.20,1.0,14.109683
3,1aue,92.0,9.874698,8.065894,9.392579,7.160846,5.37,1.0,20.379423
4,1avz,57.0,11.184179,7.011214,6.815640,5.976351,4.88,1.0,14.810961
...,...,...,...,...,...,...,...,...,...
113,2vh7,94.0,19.518273,8.485290,7.954372,7.970395,0.84,1.0,20.673717
114,2vik,126.0,15.444701,8.503297,8.990068,7.802209,6.80,1.0,25.133156
115,2vwf,56.0,10.981012,7.146772,6.652863,6.086775,2.77,1.0,14.637223
116,2vxd,54.0,6.160536,6.489503,7.918596,6.090079,7.00,1.0,14.286609


In [9]:
from sklearn.metrics import mean_squared_error
from scipy import stats

def evaluate_model(train, test, feature_columns, target_column, model_name):

    coefficients = []
    intercepts = []
    coefficient_errors = []
    intercept_errors = []
    

    X_train = train[feature_columns]
    y_train = train[target_column]

    X_test = test[feature_columns]
    y_test = test[target_column]
    
    

    X_train = sm.add_constant(X_train)
    X_test = sm.add_constant(X_test)
    
    # model = sm.WLS(y_train, X_train, weights=weights_train).fit()
    model = sm.OLS(y_train, X_train, ).fit()
    # print(model.params)

    intercepts.append(model.params['const'])
    coefficients.append(model.params.iloc[1:].values)
    coefficient_errors.append(model.bse.iloc[1:].values)
    intercept_errors.append(model.bse.iloc[0])

    avg_coefficients = np.mean(coefficients, axis=0)
    avg_intercept = np.mean(intercepts)
    avg_coefficient_errors = np.mean(coefficient_errors, axis=0)
    avg_intercept_error = np.mean(intercept_errors)
    
    formatted_coefficients = [round(coef, 3) for coef in avg_coefficients]
    formatted_coefficient_errors = [round(err, 3) for err in avg_coefficient_errors]
    
    MSE_train = mean_squared_error(y_train, model.predict(X_train))
    rpearson_train, pvalue_train = stats.pearsonr(y_train, model.predict(X_train))

    MSE_test = mean_squared_error(y_test, model.predict(X_test))
    rpearson_test, pvalue_test = stats.pearsonr(y_test, model.predict(X_test))
    
    result = {
        "Model": model_name,
        "Feature Columns": ", ".join(feature_columns),
        "Target Column": target_column,

        "Coefficients": formatted_coefficients,
        "Coefficient Errors": formatted_coefficient_errors,
        "Intercept": round(avg_intercept, 3),
        "Intercept Error": round(avg_intercept_error, 3),

        "MSE train": round(MSE_train, 3),
        "MSE test": round(MSE_test, 3),

        "R train": round(rpearson_train, 3),
        "R test": round(rpearson_test, 3),

        "p-value train": round(pvalue_train, 3),
        "p-value test": round(pvalue_test, 3),

    }
    return result


models = [
    {"model_name": "length_energy", "features": ['Length, aa'], "target": 'Energy, kcal/mol'},
    {"model_name": "CO_energy", "features": ['CO'], "target": 'Energy, kcal/mol'},
    {"model_name": "logCT_logEnergy", "features": ['log(P)', 'log(S)', 'log(X)'], "target": 'log(Energy, kcal/mol)'},

    {"model_name": "length_logkf", "features": ['Length, aa'], "target": 'ln(kf)_H2O'},
    {"model_name": "CO_logkf", "features": ['CO'], "target": 'ln(kf)_H2O'},
    {"model_name": "logCT_logkf", "features": ['log(P)', 'log(S)', 'log(X)'], "target": 'ln(kf)_H2O'},

    {"model_name": "length_logku", "features": ['Length, aa'], "target": 'ln(ku)_H2O'},
    {"model_name": "CO_logku", "features": ['CO'], "target": 'ln(ku)_H2O'},
    {"model_name": "logCT_logku", "features": ['log(P)', 'log(S)', 'log(X)'], "target": 'ln(ku)_H2O'},
]

results = []

#split data frame on test and train proteins structures
#1)
unique_proteins = KPro_df_VA_avg["PDB"].unique()
train_proteins, test_proteins = train_test_split(unique_proteins, test_size=0.5, random_state=3724761)
train = KPro_df_VA_avg[KPro_df_VA_avg["PDB"].isin(train_proteins)]
test = KPro_df_VA_avg[KPro_df_VA_avg["PDB"].isin(test_proteins)]
#2)
unique_proteins = merged_df_VA["PDB"].unique()
train_proteins, test_proteins = train_test_split(unique_proteins, test_size=0.5, random_state=6413468)
merged_train = merged_df_VA[merged_df_VA["PDB"].isin(train_proteins)]
merged_test = merged_df_VA[merged_df_VA["PDB"].isin(test_proteins)]




#train the models
for model in models:
    print(model)
    if model["target"] != 'ln(kf)_H2O': 
        result = evaluate_model(train,test,
                                feature_columns = model["features"],
                                target_column = model["target"],
                                model_name = model["model_name"])
    else:
        result = evaluate_model(merged_train,merged_test,
                                feature_columns = model["features"],
                                target_column = model["target"],
                                model_name = model["model_name"])
    results.append(result)

merged_regression_VA = pd.DataFrame(results)
merged_regression_VA['params'] = merged_regression_VA.apply(lambda row: row['Coefficients'] + [row['Intercept']], axis=1)
merged_regression_VA['name_list'] = [
                'Length, aa vs Energy',
                'Contact Order  vs Energy',
                'Log P, S, X vs logEnergy',

                'Length, aa vs log folding rate',
                'Contact Order vs log folding rate',
                'Log P, S, X vs log folding rate',
                
                'Length, aa vs log unfolding rate',
                'Contact Order  vs log unfolding rate',
                'Log P, S, X vs log unfolding rate',
]
merged_regression_VA

{'model_name': 'length_energy', 'features': ['Length, aa'], 'target': 'Energy, kcal/mol'}
{'model_name': 'CO_energy', 'features': ['CO'], 'target': 'Energy, kcal/mol'}
{'model_name': 'logCT_logEnergy', 'features': ['log(P)', 'log(S)', 'log(X)'], 'target': 'log(Energy, kcal/mol)'}
{'model_name': 'length_logkf', 'features': ['Length, aa'], 'target': 'ln(kf)_H2O'}
{'model_name': 'CO_logkf', 'features': ['CO'], 'target': 'ln(kf)_H2O'}
{'model_name': 'logCT_logkf', 'features': ['log(P)', 'log(S)', 'log(X)'], 'target': 'ln(kf)_H2O'}
{'model_name': 'length_logku', 'features': ['Length, aa'], 'target': 'ln(ku)_H2O'}
{'model_name': 'CO_logku', 'features': ['CO'], 'target': 'ln(ku)_H2O'}
{'model_name': 'logCT_logku', 'features': ['log(P)', 'log(S)', 'log(X)'], 'target': 'ln(ku)_H2O'}


,Model,Feature Columns,Target Column,Coefficients,Coefficient Errors,Intercept,Intercept Error,MSE train,MSE test,R train,R test,p-value train,p-value test,params,name_list
0,length_energy,"Length, aa","Energy, kcal/mol",[0.083],[0.017],-1.475,1.381,3.390,2.109,0.706,0.762,0.0,0.000,"[0.083, -1.475]","Length, aa vs Energy"
1,CO_energy,CO,"Energy, kcal/mol",[0.399],[0.097],0.498,1.142,3.903,3.195,0.650,0.553,0.0,0.003,"[0.399, 0.498]",Contact Order vs Energy
2,logCT_logEnergy,"log(P), log(S), log(X)","log(Energy, kcal/mol)","[0.342, 0.089, 0.139]","[0.324, 0.124, 0.408]",-2.604,0.703,0.128,0.105,0.784,0.742,0.0,0.000,"[0.342, 0.089, 0.139, -2.604]","Log P, S, X vs logEnergy"
3,length_logkf,"Length, aa",ln(kf)_H2O,[-0.037],[0.009],7.976,1.011,9.218,13.898,0.466,0.522,0.0,0.000,"[-0.037, 7.976]","Length, aa vs log folding rate"
4,CO_logkf,CO,ln(kf)_H2O,[-0.378],[0.057],9.220,0.815,6.622,9.456,0.661,0.744,0.0,0.000,"[-0.378, 9.22]",Contact Order vs log folding rate
5,logCT_logkf,"log(P), log(S), log(X)",ln(kf)_H2O,"[-1.862, 1.794, -1.842]","[0.959, 0.541, 1.306]",16.216,2.675,6.782,9.816,0.651,0.703,0.0,0.000,"[-1.862, 1.794, -1.842, 16.216]","Log P, S, X vs log folding rate"
6,length_logku,"Length, aa",ln(ku)_H2O,[-0.197],[0.035],12.375,2.773,13.670,15.812,0.762,0.720,0.0,0.000,"[-0.197, 12.375]","Length, aa vs log unfolding rate"
7,CO_logku,CO,ln(ku)_H2O,[-1.12],[0.156],9.610,1.835,10.084,8.847,0.831,0.857,0.0,0.000,"[-1.12, 9.61]",Contact Order vs log unfolding rate
8,logCT_logku,"log(P), log(S), log(X)",ln(ku)_H2O,"[-4.128, 2.137, -3.662]","[3.195, 1.224, 4.027]",34.068,6.944,12.519,10.017,0.785,0.865,0.0,0.000,"[-4.128, 2.137, -3.662, 34.068]","Log P, S, X vs log unfolding rate"


In [ ]:
merged_regression_VA.to_csv(f'{work_dir_path}\merged_regression_VA_v1.csv')

In [10]:
def pred_dG(spx_vec, param):
    dG = np.dot(spx_vec, param[:-1]) + param[-1]
    return dG

data = {
    "y_measured_energy": test['Energy, kcal/mol'],
    "y_measured_log10ku": test['ln(ku)_H2O'],

    "y_pred_length_energy": (pred_dG(test[['Length, aa']], param = np.array(merged_regression_VA.loc[merged_regression_VA["Model"]=="length_energy",'params'].values[0]))),
    "y_pred_CO_energy": (pred_dG(test[['CO']], param = np.array(merged_regression_VA.loc[merged_regression_VA["Model"]=="CO_energy","params"].values[0]))),
    "y_pred_logCT_logEnergy": np.exp(pred_dG(test[['log(P)', 'log(S)', 'log(X)']], param = np.array(merged_regression_VA.loc[merged_regression_VA["Model"]=="logCT_logEnergy","params"].values[0]))),

    "y_pred_length_logku": pred_dG(test[['Length, aa']], param = np.array(merged_regression_VA.loc[merged_regression_VA["Model"]=="length_logku","params"].values[0])),
    "y_pred_CO_logku": pred_dG(test[['CO']], param = np.array(merged_regression_VA.loc[merged_regression_VA["Model"]=="CO_logku","params"].values[0])),
    "y_pred_logCT_logku": pred_dG(test[['log(P)', 'log(S)', 'log(X)']], param = np.array(merged_regression_VA.loc[merged_regression_VA["Model"]=="logCT_logku","params"].values[0])),
}

#Take the mean of values for structures of a protein
df_lineair_regression1 = pd.DataFrame(data)
df_lineair_regression1['PDB'] = test['PDB']
df_lineair_regression1

,y_measured_energy,y_measured_log10ku,y_pred_length_energy,y_pred_CO_energy,y_pred_logCT_logEnergy,y_pred_length_logku,y_pred_CO_logku,y_pred_logCT_logku,PDB
2,4.610159,4.220000,3.5050,2.660045,3.137639,0.5550,3.541100,1.887715,1bdc
6,4.830727,-2.626667,4.6670,4.462202,3.637805,-2.2030,-1.517585,-0.225061,1div
7,4.079407,-2.730000,4.5010,4.745461,3.894656,-1.8090,-2.312698,-2.402265,1dkt
8,2.374496,5.830000,1.5960,2.927049,1.691429,5.0860,2.791618,4.245147,1e0l
10,5.640457,-8.280000,7.4060,8.047712,7.963526,-8.7040,-11.582175,-10.281415,1fkb
11,6.138144,-4.020000,4.0030,5.104522,4.310192,-0.6270,-3.320589,-4.614439,1g6p
13,1.007224,6.700000,1.8450,3.353306,2.163726,4.4950,1.595106,1.055329,1jmq
14,3.074997,-2.720000,3.3390,5.000959,3.984129,0.9490,-3.029885,-3.834295,1jo8
15,11.547532,-12.050000,11.0580,8.127617,9.802235,-17.3720,-11.806469,-10.969561,1k0s
16,4.170181,-8.050000,5.7460,6.847555,5.093954,-4.7640,-8.213313,-7.530676,1k8m


In [11]:
def pred_dG(spx_vec, param):
    dG = np.dot(spx_vec, param[:-1]) + param[-1]
    return dG

data = {
    "y_measured_log10kf": merged_test['ln(kf)_H2O'],

    "y_pred_length_logkf": pred_dG(merged_test[['Length, aa']], param = np.array(merged_regression_VA.loc[merged_regression_VA["Model"]=="length_logkf","params"].values[0])),
    "y_pred_CO_logkf": pred_dG(merged_test[['CO']], param = np.array(merged_regression_VA.loc[merged_regression_VA["Model"]=="CO_logkf","params"].values[0])),
    "y_pred_logCT_logkf": pred_dG(merged_test[['log(P)', 'log(S)', 'log(X)']], param = np.array(merged_regression_VA.loc[merged_regression_VA["Model"]=="logCT_logkf","params"].values[0])),
    
}

#Take the mean of values for structures of a protein
df_lineair_regression2 = pd.DataFrame(data)
df_lineair_regression2['PDB'] = merged_test['PDB']
df_lineair_regression2

,y_measured_log10kf,y_pred_length_logkf,y_pred_CO_logkf,y_pred_logCT_logkf,PDB
0,1.130000,2.389000,4.435523,4.183818,1a6n
1,0.690000,3.425000,3.410694,3.229179,1adw
7,6.950000,5.645000,6.329645,7.230578,1bf4
8,3.000000,3.906000,3.299065,3.098687,1bfe
9,2.560000,3.980000,4.586016,4.110169,1bni
11,3.470000,4.757000,5.216565,4.332955,1brs
12,1.790000,4.424000,6.988430,7.653815,1cdc
14,8.850000,6.607000,7.008373,6.186690,1e0m
17,10.590000,5.978000,6.437204,5.319239,1enh
18,4.040000,-8.193000,1.266199,2.421341,1fmk


#### Plot all 3

In [80]:
    

fig = make_subplots(rows=1, cols=3, horizontal_spacing= 0.1 )
font_size = 24
color_list = ['Length, aa','Contact Order','CT parameters',]
colors = ['blue', 'red', 'green']

######################################## Plot energy #########################################
# This code plots the CT parameters and contact order in the same plot
predicted_list = ['y_pred_length_energy', 'y_pred_CO_energy','y_pred_logCT_logEnergy']
experimental_data = 'y_measured_energy'



axis_min = np.min(df_lineair_regression1[predicted_list+[experimental_data]]) // 1
axis_max = -((-np.max(df_lineair_regression1[predicted_list+[experimental_data]])) // 1 )

axis_range = [axis_min, axis_max]



for i, (model_data, label, color) in enumerate(zip(predicted_list, color_list, colors)):

    # Linear regression
    slope, intercept, rvalue, _, _ = stats.linregress(df_lineair_regression1[experimental_data], df_lineair_regression1[model_data])


    fig.add_trace(go.Scatter(
        x=df_lineair_regression1[experimental_data],
        y= df_lineair_regression1[model_data],
        mode="markers",
        marker_symbol='x',
        marker=dict(color=color, size=8, opacity=0.8,
                    line=dict(color="black", width=1)),
        name=f"{label}",
        
        ),
        row=1, col=1)

    x_range = np.linspace(axis_range[0],axis_range[1], 20)
    y_pred = slope * x_range + intercept
    fig.add_trace(go.Scatter(
        x=x_range,
        y=y_pred,
        mode="lines",
        line=dict(color=color, width=3),
        showlegend=False,
        
    ),
    row=1, col=1)

#Calculate statistical parameters and add as anotations to the plot
for i, (model_data, label, color) in enumerate(zip(predicted_list, color_list, colors)):
    MSE = mean_squared_error(df_lineair_regression1[experimental_data], df_lineair_regression1[model_data])
    rpearson, pvalue = stats.pearsonr(df_lineair_regression1[experimental_data], df_lineair_regression1[model_data])

    #number of symbols in sign
    
    n_s1 = len(f"R={rpearson:.2f} ")
    n_s2 = len(f"p-value={pvalue:.2f} ")
    n_s3 = len(f"MSE = {MSE:.2f} ")

    fig.add_annotation(
        x = axis_range[0] + (axis_range[1]-axis_range[0]) * (n_s1 * 0. / 18),
        y = axis_range[1] - (axis_range[1]-axis_range[0])/8*(i+0.5),
        text=f"R={rpearson:.2f}",
        showarrow=False,
        font=dict(size=font_size*0.8, color=color),
        xanchor="left",
        bgcolor="rgba(255, 255, 255, 0.7)",
        row=1, col=1
    )
    # fig.add_annotation(
    #     x=axis_range[0] + (axis_range[1]-axis_range[0]) * ((n_s1 + n_s2 * 0.) / n_s),
    #     y=axis_range[1] - (axis_range[1]-axis_range[0])/14*(i+0.5),
    #     text=f"p-value={pvalue:.2f}",
    #     showarrow=False,
    #     font=dict(size=font_size, color=color),
    #     xanchor="left",
    #     bgcolor="rgba(255, 255, 255, 0.7)",
    #     row=1, col=1
    # )    
    fig.add_annotation(
        x=axis_range[0] + (axis_range[1]-axis_range[0]) * ((n_s1 + n_s3 * 0.) / 18),
        y=axis_range[1] - (axis_range[1]-axis_range[0])/8*(i+0.5),
        text=f"MSE = {MSE:.2f}",
        showarrow=False,
        font=dict(size=font_size*0.8, color=color),
        xanchor="left",
        bgcolor="rgba(255, 255, 255, 0.7)",
        row=1, col=1
    )

#add diagonal line
fig.add_trace(go.Scatter(
    x=axis_range,
    y=axis_range,
    mode='lines',
    line=dict(color='black', width=1),
    showlegend=False,
    ),
    row=1, col=1
    )


#Set axis names and ranges
fig.update_xaxes(
    title_text="∆𝐺<sub>exp</sub>(kcal/mol)",
    range=axis_range,
    dtick=3,
    title_font=dict(size=font_size),
    tickfont=dict(size=font_size),
    showline=True,
    linecolor="black",
    mirror=True,
    row=1, col=1
)

fig.update_yaxes(
    title_text="∆𝐺<sub>pred</sub>(kcal/mol)",
    range=axis_range,
    dtick=3,
    title_standoff=0,
    title_font=dict(size=font_size),
    tickfont=dict(size=font_size),
    showline=True,
    linecolor="black",
    mirror=True,
    row=1, col=1
)

################################################################################################################################################################

######################################## Plot kf #########################################
# This code plots the CT parameters and contact order in the same plot
predicted_list = ['y_pred_length_logkf', 'y_pred_CO_logkf','y_pred_logCT_logkf']

experimental_data = 'y_measured_log10kf'

axis_min = np.min(df_lineair_regression2[predicted_list+[experimental_data]]) // 1
axis_max = -((-np.max(df_lineair_regression2[predicted_list+[experimental_data]])) // 1 )

axis_range = [axis_min, 20]

for i, (model_data, label, color) in enumerate(zip(predicted_list, color_list, colors)):

    # Linear regression
    slope, intercept, rvalue, _, _ = stats.linregress(df_lineair_regression2[experimental_data], df_lineair_regression2[model_data])


    fig.add_trace(go.Scatter(
        x=df_lineair_regression2[experimental_data],
        y= df_lineair_regression2[model_data],
        mode="markers",
        marker_symbol='x',
        marker=dict(color=color, size=8, opacity=0.8,
                    line=dict(color="black", width=1)),
        name=f"{label}",
        showlegend=False,
        ),
        row=1, col=2)

    x_range = np.linspace(axis_range[0],axis_range[1], 20)
    y_pred = slope * x_range + intercept
    fig.add_trace(go.Scatter(
        x=x_range,
        y=y_pred,
        mode="lines",
        line=dict(color=color, width=3),
        showlegend=False,
        
    ),
    row=1, col=2)

#Calculate statistical parameters and add as anotations to the plot
for i, (model_data, label, color) in enumerate(zip(predicted_list, color_list, colors)):
    MSE = mean_squared_error(df_lineair_regression2[experimental_data], df_lineair_regression2[model_data])
    rpearson, pvalue = stats.pearsonr(df_lineair_regression2[experimental_data], df_lineair_regression2[model_data])

    #number of symbols in sign
    
    n_s1 = len(f"R={rpearson:.2f} ")
    n_s2 = len(f"p-value={pvalue:.2f} ")
    n_s3 = len(f"MSE = {MSE:.2f} ")

    fig.add_annotation(
        x = axis_range[0] + (axis_range[1]-axis_range[0]) * (n_s1 * 0. / 18),
        y = axis_range[1] - (axis_range[1]-axis_range[0])/8*(i+0.5),
        text=f"R={rpearson:.2f}",
        showarrow=False,
        font=dict(size=font_size*0.8, color=color),
        xanchor="left",
        bgcolor="rgba(255, 255, 255, 0.7)",
        row=1, col=2
    )
    # fig.add_annotation(
    #     x=axis_range[0] + (axis_range[1]-axis_range[0]) * ((n_s1 + n_s2 * 0.) / n_s),
    #     y=axis_range[1] - (axis_range[1]-axis_range[0])/14*(i+0.5),
    #     text=f"p-value={pvalue:.2f}",
    #     showarrow=False,
    #     font=dict(size=font_size, color=color),
    #     xanchor="left",
    #     bgcolor="rgba(255, 255, 255, 0.7)",
    #     row=1, col=2
    # )    
    fig.add_annotation(
        x=axis_range[0] + (axis_range[1]-axis_range[0]) * ((n_s1 + n_s3 * 0.) / 18),
        y=axis_range[1] - (axis_range[1]-axis_range[0])/8*(i+0.5),
        text=f"MSE = {MSE:.2f}",
        showarrow=False,
        font=dict(size=font_size*0.8, color=color),
        xanchor="left",
        bgcolor="rgba(255, 255, 255, 0.7)",
        row=1, col=2
    )

#add diagonal line
fig.add_trace(go.Scatter(
    x=axis_range,
    y=axis_range,
    mode='lines',
    line=dict(color='black', width=1),
    showlegend=False,
    ),
    row=1, col=2
    )


#Set axis names and ranges
fig.update_xaxes(
    title_text="ln k<sub>f,exp</sub>(kcal/mol)",
    range=axis_range,
    dtick=6,
    title_font=dict(size=font_size),
    tickfont=dict(size=font_size),
    showline=True,
    linecolor="black",
    mirror=True,
    row=1, col=2
)

fig.update_yaxes(
    title_text="ln k<sub>f,pred</sub>(kcal/mol)",
    range=axis_range,
    dtick=6,
    title_standoff=0,
    title_font=dict(size=font_size),
    tickfont=dict(size=font_size),
    showline=True,
    linecolor="black",
    mirror=True,
    row=1, col=2
)

################################################################################################################################################################

######################################## Plot ku #########################################
# This code plots the CT parameters and contact order in the same plot
predicted_list = ['y_pred_length_logku', 'y_pred_CO_logku','y_pred_logCT_logku']

experimental_data = 'y_measured_log10ku'

axis_min = np.min(df_lineair_regression1[predicted_list+[experimental_data]]) // 1
axis_max = -((-np.max(df_lineair_regression1[predicted_list+[experimental_data]])) // 1 )

axis_range = [axis_min, 14]

for i, (model_data, label, color) in enumerate(zip(predicted_list, color_list, colors)):

    # Linear regression
    slope, intercept, rvalue, _, _ = stats.linregress(df_lineair_regression1[experimental_data], df_lineair_regression1[model_data])


    fig.add_trace(go.Scatter(
        x=df_lineair_regression1[experimental_data],
        y= df_lineair_regression1[model_data],
        mode="markers",
        marker_symbol='x',
        marker=dict(color=color, size=8, opacity=0.8,
                    line=dict(color="black", width=1)),
        name=f"{label}",
        showlegend=False,
        ),
        row=1, col=3)

    x_range = np.linspace(axis_range[0],axis_range[1], 20)
    y_pred = slope * x_range + intercept
    fig.add_trace(go.Scatter(
        x=x_range,
        y=y_pred,
        mode="lines",
        line=dict(color=color, width=3),
        showlegend=False,
        
    ),
    row=1, col=3)

#Calculate statistical parameters and add as anotations to the plot
for i, (model_data, label, color) in enumerate(zip(predicted_list, color_list, colors)):
    MSE = mean_squared_error(df_lineair_regression1[experimental_data], df_lineair_regression1[model_data])
    rpearson, pvalue = stats.pearsonr(df_lineair_regression1[experimental_data], df_lineair_regression1[model_data])

    #number of symbols in sign
    
    n_s1 = len(f"R={rpearson:.2f} ")
    n_s2 = len(f"p-value={pvalue:.2f} ")
    n_s3 = len(f"MSE = {MSE:.2f} ")

    fig.add_annotation(
        x = axis_range[0] + (axis_range[1]-axis_range[0]) * (n_s1 * 0. / 18),
        y = axis_range[1] - (axis_range[1]-axis_range[0]) / 8 * (i+0.5),
        text=f"R={rpearson:.2f}",
        showarrow=False,
        font=dict(size=font_size*0.8, color=color),
        xanchor="left",
        bgcolor="rgba(255, 255, 255, 0.7)",
        row=1, col=3
    )
    # fig.add_annotation(
    #     x=axis_range[0] + (axis_range[1]-axis_range[0]) * ((n_s1 + n_s2 * 0.) / n_s),
    #     y=axis_range[1] - (axis_range[1]-axis_range[0])/14*(i+0.5),
    #     text=f"p-value={pvalue:.2f}",
    #     showarrow=False,
    #     font=dict(size=font_size, color=color),
    #     xanchor="left",
    #     bgcolor="rgba(255, 255, 255, 0.7)",
    #     row=1, col=3
    # )    
    fig.add_annotation(
        x=axis_range[0] + (axis_range[1]-axis_range[0]) * ((n_s1 + n_s3 * 0.) / 18),
        y=axis_range[1] - (axis_range[1]-axis_range[0]) / 8 * (i+0.5),
        text=f"MSE = {MSE:.2f}",
        showarrow=False,
        font=dict(size=font_size*0.8, color=color),
        xanchor="left",
        bgcolor="rgba(255, 255, 255, 0.7)",
        row=1, col=3
    )

#add diagonal line
fig.add_trace(go.Scatter(
    x=axis_range,
    y=axis_range,
    mode='lines',
    line=dict(color='black', width=1),
    showlegend=False,
    ),
    row=1, col=3
    )




#Set axis names and ranges
fig.update_xaxes(
    title_text="ln k<sub>u,exp</sub>(kcal/mol)",
    range=axis_range,
    dtick=7,
    title_font=dict(size=font_size),
    tickfont=dict(size=font_size),
    showline=True,
    linecolor="black",
    mirror=True,
    row=1, col=3
)

fig.update_yaxes(
    title_text="ln k<sub>u,pred</sub>(kcal/mol)",
    range=axis_range,
    dtick=7,
    title_standoff=0,
    title_font=dict(size=font_size),
    tickfont=dict(size=font_size),
    showline=True,
    linecolor="black",
    mirror=True,
    row=1, col=3
)

################################################################################################################################################################

#Plot tytle
fig.update_layout(
    # title_text=" Predicted free energy",
    # title_x=0.5,
    # title_font=dict(size=font_size),
    width=945,
    height=400,
    showlegend=True,
    plot_bgcolor="white",
    font=dict(family="Arial", size=font_size, color='black'),
    # margin=dict(l=200, r=100, t=80, b=100),
    # row=1, col=1
)

fig.update_layout(legend=dict(
    orientation="h",
    yanchor="bottom",
    y=1.07,
    xanchor="right",
    x=1
))


fig.write_image(f'{work_dir_path}/images/models_all_combined_test_v2.png', scale=4, 
    width=945,
    height=400,)

fig.show()



### Plotings

In [448]:
fig = make_subplots(rows=3, cols=3)

fig.add_trace(go.Scatter(x=KPro_df_VA_avg["log(P)"], y=KPro_df_VA_avg["log(Energy, kcal/mol)"],mode="markers"), row = 1, col = 1)
fig.add_trace(go.Scatter(x=KPro_df_VA_avg["log(S)"], y=KPro_df_VA_avg["log(Energy, kcal/mol)"],mode="markers"), row = 2, col = 1)
fig.add_trace(go.Scatter(x=KPro_df_VA_avg["log(X)"], y=KPro_df_VA_avg["log(Energy, kcal/mol)"],mode="markers"), row = 3, col = 1)


fig.add_trace(go.Scatter(x=merged_df_VA["log(P)"], y=merged_df_VA["ln(kf)_H2O"],mode="markers"), row = 1, col = 2)
fig.add_trace(go.Scatter(x=merged_df_VA["log(S)"], y=merged_df_VA["ln(kf)_H2O"],mode="markers"), row = 2, col = 2)
fig.add_trace(go.Scatter(x=merged_df_VA["log(X)"], y=merged_df_VA["ln(kf)_H2O"],mode="markers"), row = 3, col = 2)

fig.add_trace(go.Scatter(x=KPro_df_VA_avg["log(P)"], y=KPro_df_VA_avg["ln(ku)_H2O"],mode="markers"), row = 1, col = 3)
fig.add_trace(go.Scatter(x=KPro_df_VA_avg["log(S)"], y=KPro_df_VA_avg["ln(ku)_H2O"],mode="markers"), row = 2, col = 3)
fig.add_trace(go.Scatter(x=KPro_df_VA_avg["log(X)"], y=KPro_df_VA_avg["ln(ku)_H2O"],mode="markers"), row = 3, col = 3)


fig.update_layout(height=800, width=800,)
fig.show()


In [519]:
# Plot surface
X, Y, Z = np.mgrid[:12:20j, :12:20j, :12:20j]

param = [-1.862, 1.794, -1.842, 16.216]	 #ln(kf) parameters
vol = X*param[0] + Y*param[1] + Z*param[2] + param[3]



# Combine both value sources to compute global min/max
all_values = np.concatenate([vol.flatten(), merged_df_VA['ln(kf)_H2O'].values])
cmin, cmax = all_values.min(), all_values.max()

# 3. Create volume plot
fig = go.Figure(data=go.Volume(
    x=X.flatten(), y=Y.flatten(), z=Z.flatten(),
    value=vol.flatten(),
    isomin=cmin,
    isomax=cmax,
    opacity=0.05,
    surface_count=20,
    caps=dict(x_show=True, y_show=True, z_show=True, x_fill=1),
    colorscale='RdBu',
    cmin=min(merged_df_VA['ln(kf)_H2O'].values),
    cmax=max(merged_df_VA['ln(kf)_H2O'].values),
    showscale=False  # hides duplicate colorbar
))

# 4. Add scatter trace
fig.add_trace(go.Scatter3d(
    x=merged_df_VA['log(P)'],
    y=merged_df_VA['log(S)'],
    z=merged_df_VA['log(X)'],
    mode='markers',
    marker=dict(
        size=6,
        color=merged_df_VA['ln(kf)_H2O'],
        colorscale='RdBu',
        cmin=min(merged_df_VA['ln(kf)_H2O'].values),
        cmax=max(merged_df_VA['ln(kf)_H2O'].values),
        colorbar=dict(
            title='ln(kf)_H2O',
            # textfont=dict(size=16),
            tickfont=dict(size=16)
            ),
        showscale=True
    )
))

# 5. Layout
fig.update_layout(
    scene_camera=dict(
        up=dict(x=0, y=0, z=1),
        center=dict(x=0, y=0, z=0),
        eye=dict(x=-1.5, y=-1.5, z=1.5),
    ),
    scene=dict(
        xaxis=dict(title='log(P)', tickfont=dict(size=16), backgroundcolor="white"),
        yaxis=dict(title='log(S)', tickfont=dict(size=16), backgroundcolor="white"),
        zaxis=dict(title='log(X)', tickfont=dict(size=16), backgroundcolor="white"),
    ),
    plot_bgcolor="white",
    font=dict(family="Arial", size=16, color='black'),  # applies to legend, colorbar titles, etc.
    width=800,
    height=800
)

fig.show()